## === WORKFLOW ===

**Phase 1 (GEE) → `04_export_modis_igbp.ipynb`:**
- Lade MODIS MCD12Q1 IGBP für alle Jahre 2001–2024
- Reproject zu 500m EPSG:3035, Clippe auf wildE-Grid
- Export als 24-Band GeoTIFF → `MODIS_IGBP_2001_2024_500m_wildE.tif`

**Phase 2 (Lokal) → `04_export_modis_igbp.ipynb`:**
- Download via PyDrive + Validierung gegen Grid-Referenz
- *(Kein combined TIF — alle Dateien werden in Phase 3 separat geladen)*

**Phase 3 (Lokal) — diese Datei:**

| Schritt | Beschreibung |
|---|---|
| **0** | Lade alle Daten (Woody, MBA, MODIS IGBP, Ecoregions — separat) |
| **1** | Pre-Fire Landcover (MODIS IGBP Jahr vor Brand) |
| **2a–d** | LC-Trajektorien (Major-Klasse, IGBP-Einzelklasse, LC×FireCount, LC×Ecoregion) |
| **2e** | Eco-Trajektorien (1 Fire) |
| **2f** | Eco × Fire Count |
| **2g** | Fire Statistics (vektorisiert) |
| **2h** | Recovery-%-Index (loss-relativ, Fire Freq 1–4, +10yr) |
| **2i** | Heatmap Ecoregion × LC (Fläche km²/%) |
| **3** | LC-Visualisierungen (Plots 1–5) |
| **3b** | Stat. Tests LC (Kruskal-Wallis + Dunn, europa-weit & pro Ecoregion) |
| **3c** | Eco-Visualisierungen (E1–E10) |
| **3d** | Stat. Tests Ecoregion (Kruskal-Wallis + Dunn) |
| **4** | CSV-Export + Abschlussmeldung |
| **★** | Schnelle Visualisierung aus CSV (eine Zelle, ganz am Ende) |


## Kurzübersicht

| Phase | Wo | Was |
|---|---|---|
| **Phase 1** | GEE | MODIS MCD12Q1 IGBP Export (2001–2024) → 24-Band GeoTIFF |
| **Phase 2** | Lokal | Download + Validierung `MODIS_IGBP_2001_2024_500m_wildE.tif` (kein combined TIF) |
| **Phase 3** | Lokal | Analyse → Plots 1–5 (LC), E1–E10 (Eco), Stat. Tests, CSV-Export |

→ Details siehe Workflow-Zelle oben · Phase 1+2 → `04_export_modis_igbp.ipynb`


## === WORKFLOW ===

**Phase 1 (GEE) → `04_export_modis_igbp.ipynb`:**
- Lade MODIS MCD12Q1 IGBP für alle Jahre 2001–2024
- Reproject zu 500m EPSG:3035, Clippe auf wildE-Grid
- Export als 24-Band GeoTIFF → `MODIS_IGBP_2001_2024_500m_wildE.tif`

**Phase 2 (Lokal) → `04_export_modis_igbp.ipynb`:**
- Download via PyDrive + Validierung gegen Grid-Referenz
- *(Kein combined TIF — alle Dateien werden in Phase 3 separat geladen)*

**Phase 3 (Lokal) — diese Datei:**

| Schritt | Zell-ID | Beschreibung |
|---|---|---|
| **Config** | `#VSC-f8503248` | Pfade, Jahresbereiche, Output-Verzeichnisse |
| **0** | `#VSC-4f09bd04` | Lade alle Daten (Woody, MBA, MODIS IGBP, Ecoregions) |
| **1** | `#VSC-506cc7d7` | Pre-Fire Landcover (MODIS IGBP Jahr vor Brand) |
| **Hilfs-fkt.** | `#VSC-fac82136` | `calc_trajectory()`, `calc_trajectory_by_fire_count()` |
| **2a–d** | `#VSC-60353918` | LC-Trajektorien (Major-Klasse, IGBP-Einzelklasse, LC×FireCount, LC×Ecoregion) |
| **2e** | `#VSC-ef723a8a` | Eco-Trajektorien (1 Fire) |
| **2f** | `#VSC-3395bfae` | Eco × Fire Count |
| **2g** | `#VSC-60783a3d` | Fire Statistics (vektorisiert): Burned Area, Season Length, Fire Count |
| **2h** | `#VSC-54dd79fe` | Recovery-%-Index (loss-relativ, Fire Freq 1–4, +10yr) |
| **2i** | `#VSC-ba949c68` | Heatmap Ecoregion × LC (Fläche km²/%) |
| **3** | `#VSC-224f77fd` | LC-Visualisierungen (Plots 1–5) |
| **3b** | `#VSC-6cd2c209` | Stat. Tests LC (Kruskal-Wallis + Dunn, europa-weit & pro Ecoregion) |
| **3c** | `#VSC-397c9f3d` | Eco-Visualisierungen (E1–E10) |
| **3d** | `#VSC-0fa140ec` | Stat. Tests Ecoregion (Kruskal-Wallis + Dunn) |
| **4** | `#VSC-b1325ef2` | CSV-Export (13 Dateien) + Abschlussmeldung |
| **★** | `#VSC-6ee6fdfd` | Schnelle Visualisierung aus CSV (ohne Neuberechnung) |

> ⚠️ **Zellen `#VSC-700fe552` und `#VSC-8e4f0401` sind Duplikate von Schritt 3 und 3b → löschen!**

## Phase 3: MODIS IGBP Land Cover Analysis


In [6]:
# === PHASE 3: LANDCOVER-STRATIFIZIERTE TRAJEKTORIEN-ANALYSE ===
# Berechnet Pre-/Post-Fire Woody Cover Trajektorien stratifiziert nach MODIS IGBP Landcover

import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"

# === KONFIGURATION ===
years_woody  = list(range(1985, 2025))   # 40 Bänder (1985-2024)
years_burned = list(range(2001, 2026))   # 25 Jahre  (2001-2025)
MBA_START_YEAR = 2000                    # Erstes Jahr im MBA-Raster (Band-Index-Basis)
band_structure = [
    "burned", "count", "doy_1", "doy_2", "doy_3", "doy_4",
    "doy_min", "doy_max", "doy_span", "month_first", "month_last", "season_length"
]
modis_years = list(range(2001, 2025))    # 24 Bänder (2001-2024)

# === Separate Dateipfade (kein combined TIF) ===
woody_path  = os.path.join(workDir, r"_Runs\02_Woody_Cover\woody_cover_500m_3035.tif")
burned_path = os.path.join(workDir, r"_Runs\01_Fire_Metrics\MBA_500m_3035_DOY_season\MBA_burned_metrics_yearly_season_500m_mosaic.tif")

ecoregion_raster_path = os.path.join(workDir, r"_Runs\03_Ecoregions_EEA\ecoregions_500m_3035_MBA.tif")
ecoregion_attr_path   = os.path.join(workDir, r"_Runs\03_Ecoregions_EEA\ecoregions_attributes_with_colors.csv")

modis_path  = os.path.join(workDir, r"_Runs\04_Landcover\MODIS_GEE_tiles\MODIS_IGBP_2001_2024_500m_wildE.tif")

LC_EXCLUDE  = {'Barren/Ice'}   # zu selten/irrelevant für Europa
ECO_EXCLUDE = {'OUT'}          # "Outside" = außerhalb EEA-Studiengebiet

# === OUTPUT-VERZEICHNISSE (09_Results) ===
results_base      = os.path.join(workDir, "_Runs", "09_Results")

eco_plots_dir     = os.path.join(results_base, "Ecoregions",     "plots")
eco_csv_dir       = os.path.join(results_base, "Ecoregions",     "csv")
lc_plots_dir      = os.path.join(results_base, "Landcover",      "plots")
lc_csv_dir        = os.path.join(results_base, "Landcover",      "csv")
lcxeco_plots_dir  = os.path.join(results_base, "LC_x_Ecoregion", "plots")
lcxeco_csv_dir    = os.path.join(results_base, "LC_x_Ecoregion", "csv")

for d in [eco_plots_dir, eco_csv_dir, lc_plots_dir, lc_csv_dir, lcxeco_plots_dir, lcxeco_csv_dir]:
    os.makedirs(d, exist_ok=True)

print("=" * 70)
print("LANDCOVER-STRATIFIZIERTE TRAJEKTORIEN-ANALYSE")
print("=" * 70)
print(f"  Woody Cover : {years_woody[0]}-{years_woody[-1]}  ({len(years_woody)} Bänder)")
print(f"  MBA Burned  : {years_burned[0]}-{years_burned[-1]}  ({len(years_burned)} Jahre × {len(band_structure)} Metriken)")
print(f"  MODIS IGBP  : {modis_years[0]}-{modis_years[-1]}  ({len(modis_years)} Bänder)")
print(f"\n  Output:")
print(f"    Ecoregions:     {eco_plots_dir}")
print(f"    Landcover:      {lc_plots_dir}")
print(f"    LC×Ecoregion:   {lcxeco_plots_dir}")


LANDCOVER-STRATIFIZIERTE TRAJEKTORIEN-ANALYSE
  Woody Cover : 1985-2024  (40 Bänder)
  MBA Burned  : 2001-2025  (25 Jahre × 12 Metriken)
  MODIS IGBP  : 2001-2024  (24 Bänder)

  Output:
    Ecoregions:     \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Ecoregions\plots
    Landcover:      \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Landcover\plots
    LC×Ecoregion:   \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\LC_x_Ecoregion\plots


In [7]:
# === SCHRITT 0: LADE ALLE DATEN ===

print("\n0. LADE DATEN")
print("-" * 70)

# --- Grid-Eigenschaften aus woody_path (Referenz-Raster) ---
with rasterio.open(woody_path) as src:
    nodata    = src.nodata
    transform = src.transform
    crs       = src.crs
    height    = src.height
    width     = src.width

# --- Lade Woody Cover (alle 40 Jahre, 1985–2024) ---
print("Lade Woody Cover (1985-2024)...")
with rasterio.open(woody_path) as src:
    woody = src.read()   # Shape: (40, H, W)

print(f"  ✓ Woody Cover: {woody.shape}")

# --- Lade Burned Area: "burned"-Band pro Jahr (2001-2025) ---
# Band-Index basiert auf MBA_START_YEAR (2000), da das Raster ab Jahr 2000 vorliegt
print(f"Lade Burned Area ({years_burned[0]}-{years_burned[-1]})...")
with rasterio.open(burned_path) as src:
    burned_band_indices = [
        1 + (year - MBA_START_YEAR) * len(band_structure)
        for year in years_burned
    ]
    burned = src.read(burned_band_indices)   # Shape: (25, H, W)

print(f"  ✓ Burned Area: {burned.shape}")

# --- Lade MODIS IGBP (alle 24 Jahre, 2001–2024) ---
print("Lade MODIS IGBP (2001-2024)...")
with rasterio.open(modis_path) as src:
    modis_igbp = src.read()   # Shape: (24, H, W)

print(f"  ✓ MODIS IGBP: {modis_igbp.shape}")

# --- Lade Ecoregion-Raster & Attribute ---
print("Lade Ecoregion-Raster...")
with rasterio.open(ecoregion_raster_path) as src:
    eco_raster = src.read(1)

ecoregion_df = pd.read_csv(ecoregion_attr_path)

eco_mapping = {}
for _, row in ecoregion_df.iterrows():
    eco_mapping[row['NUMERIC_ID']] = {
        'code':      row['code'],
        'name':      row['name'],
        'hex_color': row['hex_color']
    }

unique_eco_ids = np.array([
    eid for eid in np.unique(eco_raster[eco_raster > 0]).astype(int)
    if eco_mapping[eid]['code'] not in ECO_EXCLUDE
])
print(f"  ✓ Ecoregions: {len(unique_eco_ids)} Regionen (exkl. {ECO_EXCLUDE})")
print(f"\n✓ Alle Daten geladen!")



0. LADE DATEN
----------------------------------------------------------------------
Lade Woody Cover (1985-2024)...
  ✓ Woody Cover: (40, 9660, 10483)
Lade Burned Area (2001-2025)...
  ✓ Burned Area: (25, 9660, 10483)
Lade MODIS IGBP (2001-2024)...
  ✓ MODIS IGBP: (24, 9660, 10483)
Lade Ecoregion-Raster...
  ✓ Ecoregions: 11 Regionen (exkl. {'OUT'})

✓ Alle Daten geladen!


In [8]:
# === SCHRITT 1: BESTIMME PRE-FIRE LANDCOVER PRO PIXEL ===
# Für jedes Pixel das MODIS IGBP Landcover VOR dem ersten Brand verwenden

print("\n1. BESTIMME PRE-FIRE LANDCOVER PRO PIXEL")
print("-" * 70)

# IGBP Klassen-Definition
igbp_classes = {
    1: "Evergreen Needleleaf Forests",
    2: "Evergreen Broadleaf Forests",
    3: "Deciduous Needleleaf Forests",
    4: "Deciduous Broadleaf Forests",
    5: "Mixed Forests",
    6: "Closed Shrublands",
    7: "Open Shrublands",
    8: "Woody Savannas",
    9: "Savannas",
    10: "Grasslands",
    11: "Permanent Wetlands",
    12: "Cropland",
    13: "Urban and Built-up Lands",
    14: "Cropland/Natural Vegetation Mosaics",
    15: "Permanent Snow and Ice",
    16: "Barren",
    17: "Water Bodies"
}

# Lookup-Array für Major-Klassen (Index = IGBP Code, Wert = Major-Klassen-ID)
MAJOR_LC_NAMES = ['', 'Forests', 'Shrublands', 'Open Woodland', 'Grasslands',
                  'Wetlands', 'Croplands', 'Urban', 'Barren/Ice', 'Water', 'Other']
igbp_to_major_id = np.array([
    0,   # 0: NoData
    1,   # 1: Evergreen Needleleaf → Forests
    1,   # 2: Evergreen Broadleaf → Forests
    1,   # 3: Deciduous Needleleaf → Forests
    1,   # 4: Deciduous Broadleaf → Forests
    1,   # 5: Mixed Forests → Forests
    2,   # 6: Closed Shrublands → Shrublands
    2,   # 7: Open Shrublands → Shrublands
    3,   # 8: Woody Savannas → Open Woodland
    3,   # 9: Savannas → Open Woodland
    4,   # 10: Grasslands
    5,   # 11: Wetlands
    6,   # 12: Cropland → Croplands
    7,   # 13: Urban
    6,   # 14: Cropland/Veg Mosaic → Croplands
    8,   # 15: Snow/Ice → Barren/Ice
    8,   # 16: Barren → Barren/Ice
    9,   # 17: Water
], dtype=np.uint8)

# --- Berechne Anzahl Brände pro Pixel ---
fire_counts = np.sum(burned == 1, axis=0)
has_fire = fire_counts > 0

# --- Bestimme erstes Brandjahr-Index pro Pixel (vektorisiert) ---
first_fire_idx = np.argmax(burned == 1, axis=0)  # Index in years_burned

# --- Berechne MODIS-Index für Pre-Fire LC (vektorisiert) ---
print("Berechne Pre-Fire Landcover Zuordnung (vektorisiert)...")

# Brandjahr pro Pixel → MODIS-Ziel-Jahr = Brandjahr - 1
fire_year_values = np.array(years_burned)[first_fire_idx]  # shape: (H, W)
target_years = fire_year_values - 1

# Clippe auf MODIS-Verfügbarkeit [2001, 2024]
modis_idx = np.clip(target_years - modis_years[0], 0, len(modis_years) - 1)

# Extrahiere MODIS LC per Fancy Indexing
rows, cols = np.where(has_fire)
pre_fire_lc = np.zeros((height, width), dtype=np.uint8)
pre_fire_lc[rows, cols] = modis_igbp[modis_idx[rows, cols], rows, cols]

# Filtere ungültige Werte (nur 1-17 behalten)
invalid = (pre_fire_lc < 1) | (pre_fire_lc > 17)
pre_fire_lc[invalid] = 0

# --- Erstelle Major-Klassen-ID Raster (vektorisiert via Lookup) ---
pre_fire_lc_major_id = igbp_to_major_id[pre_fire_lc]  # shape: (H, W), dtype: uint8

# --- Barren/Ice ausschließen (Major ID 8) ---
pre_fire_lc_major_id[pre_fire_lc_major_id == 8] = 0

# Statistik
unique_lc, lc_counts = np.unique(pre_fire_lc[pre_fire_lc > 0], return_counts=True)
print(f"\n✓ Pre-Fire Landcover zugeordnet für {np.sum(pre_fire_lc > 0):,} gebrannte Pixel")
print(f"\nVerteilung der Pre-Fire IGBP Klassen:")
for lc, cnt in zip(unique_lc, lc_counts):
    pct = cnt / lc_counts.sum() * 100
    print(f"  {lc:2d} ({igbp_classes.get(lc, '?'):40s}): {cnt:>8,} Pixel ({pct:.1f}%)")

# Major-Klassen Statistik
unique_major, major_counts = np.unique(pre_fire_lc_major_id[pre_fire_lc_major_id > 0], return_counts=True)
print(f"\nVerteilung der Major Landcover-Klassen (exkl. Barren/Ice):")
for mid, cnt in zip(unique_major, major_counts):
    pct = cnt / major_counts.sum() * 100
    print(f"  {MAJOR_LC_NAMES[mid]:20s}: {cnt:>8,} Pixel ({pct:.1f}%)")



1. BESTIMME PRE-FIRE LANDCOVER PRO PIXEL
----------------------------------------------------------------------
Berechne Pre-Fire Landcover Zuordnung (vektorisiert)...

✓ Pre-Fire Landcover zugeordnet für 4,300,774 gebrannte Pixel

Verteilung der Pre-Fire IGBP Klassen:
   1 (Evergreen Needleleaf Forests            ):   45,746 Pixel (1.1%)
   2 (Evergreen Broadleaf Forests             ):    6,240 Pixel (0.1%)
   3 (Deciduous Needleleaf Forests            ):       93 Pixel (0.0%)
   4 (Deciduous Broadleaf Forests             ):   54,896 Pixel (1.3%)
   5 (Mixed Forests                           ):   74,072 Pixel (1.7%)
   6 (Closed Shrublands                       ):    1,647 Pixel (0.0%)
   7 (Open Shrublands                         ):   14,483 Pixel (0.3%)
   8 (Woody Savannas                          ):  202,348 Pixel (4.7%)
   9 (Savannas                                ):  246,406 Pixel (5.7%)
  10 (Grasslands                              ): 1,064,163 Pixel (24.7%)
  11 (Permanent W

In [9]:
# === HILFSFUNKTIONEN: TRAJEKTORIE-BERECHNUNG ===
# Wird von Eco- UND LC-Analyse genutzt → muss vor beiden stehen

def calc_trajectory(woody, burned, mask, years_woody, years_burned, nodata, 
                    rel_years=range(-5, 11)):
    """
    Berechnet Pre-/Post-Fire Woody Cover Trajektorie für maskierte Pixel.
    Verwendet NUR Pixel mit GENAU 1 Brandereignis.
    """
    single_fire_mask = (fire_counts == 1) & mask
    n_pixels = np.sum(single_fire_mask)
    if n_pixels < 30:
        return None

    offset = years_burned[0] - years_woody[0]  # 2001 - 1985 = 16

    y_all, x_all = np.where(single_fire_mask)
    fire_idx_subset = first_fire_idx[y_all, x_all]

    trajectory, trajectory_std, trajectory_n = [], [], []

    for rel_year in rel_years:
        woody_band_idx = fire_idx_subset + rel_year + offset - 1
        valid = (woody_band_idx >= 0) & (woody_band_idx < len(years_woody))

        if np.sum(valid) == 0:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)
            trajectory_n.append(0)
            continue

        values = woody[woody_band_idx[valid], y_all[valid], x_all[valid]].astype(float)
        values = values[(values != nodata) & (~np.isnan(values))]

        if len(values) > 0:
            trajectory.append(np.nanmean(values))
            trajectory_std.append(np.nanstd(values))
            trajectory_n.append(len(values))
        else:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)
            trajectory_n.append(0)

    return {'trajectory': trajectory, 'std': trajectory_std,
            'n_per_year': trajectory_n, 'n_pixels': int(n_pixels)}


def calc_trajectory_by_fire_count(woody, burned, mask, years_woody, years_burned,
                                   nodata, fire_count_value, rel_years=range(-5, 11)):
    """
    Berechnet Trajektorie für Pixel mit bestimmter Brandanzahl.
    """
    specific_mask = (fire_counts == fire_count_value) & mask
    n_pixels = np.sum(specific_mask)
    if n_pixels < 30:
        return None

    offset = years_burned[0] - years_woody[0]

    y_all, x_all = np.where(specific_mask)
    fire_idx_subset = first_fire_idx[y_all, x_all]

    trajectory, trajectory_std = [], []

    for rel_year in rel_years:
        woody_band_idx = fire_idx_subset + rel_year + offset - 1
        valid = (woody_band_idx >= 0) & (woody_band_idx < len(years_woody))

        if np.sum(valid) == 0:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)
            continue

        values = woody[woody_band_idx[valid], y_all[valid], x_all[valid]].astype(float)
        values = values[(values != nodata) & (~np.isnan(values))]

        if len(values) > 0:
            trajectory.append(np.nanmean(values))
            trajectory_std.append(np.nanstd(values))
        else:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)

    return {'trajectory': trajectory, 'std': trajectory_std, 'n_pixels': int(n_pixels)}


rel_years = list(range(-5, 11))  # -5 bis +10 Jahre relativ zum Brand

# Major-Klassen für spätere LC-Analyse vorbereiten
SKIP_MAJOR_IDS = {0, 7, 9, 10}  # NoData, Urban, Water, Other
major_classes_present = sorted(set(
    int(mid) for mid in unique_major if mid not in SKIP_MAJOR_IDS
))
print(f"✓ Trajektorie-Funktionen geladen. {len(major_classes_present)} Major LC Klassen vorhanden: "
      f"{[MAJOR_LC_NAMES[m] for m in major_classes_present]}")


✓ Trajektorie-Funktionen geladen. 6 Major LC Klassen vorhanden: ['Forests', 'Shrublands', 'Open Woodland', 'Grasslands', 'Wetlands', 'Croplands']


In [17]:
# === SCHRITT 2: BERECHNE TRAJEKTORIEN NACH LANDCOVER ===

print("\n2. BERECHNE TRAJEKTORIEN NACH LANDCOVER")
print("-" * 70)
print(f"Analysiere {len(major_classes_present)} Major LC Klassen: "
      f"{[MAJOR_LC_NAMES[m] for m in major_classes_present]}")

# === 2a: Trajektorien nach IGBP Major-Klasse (1 Fire) ===
print("\n2a. Trajektorien nach IGBP Major-Klasse (1 Fire Event)...")

lc_major_results = {}

for major_id in tqdm(major_classes_present, desc="Major LC Klassen"):
    lc_name = MAJOR_LC_NAMES[major_id]
    mask = (pre_fire_lc_major_id == major_id)
    result = calc_trajectory(woody, burned, mask, years_woody, years_burned, nodata, rel_years)
    if result is not None:
        lc_major_results[lc_name] = result
        print(f"  ✓ {lc_name}: {result['n_pixels']:,} Pixel")
    else:
        print(f"  ✗ {lc_name}: zu wenige Pixel (<30)")

# === 2b: Trajektorien nach IGBP Einzelklasse (1 Fire) ===
print("\n2b. Trajektorien nach IGBP Einzelklasse (1 Fire Event)...")

lc_detail_results = {}

for lc_code in tqdm(unique_lc, desc="IGBP Klassen"):
    if lc_code in [13, 15, 16, 17]:  # Skip Urban, Ice, Barren, Water
        continue
    mask = (pre_fire_lc == lc_code)
    result = calc_trajectory(woody, burned, mask, years_woody, years_burned, nodata, rel_years)
    if result is not None:
        lc_detail_results[int(lc_code)] = result
        print(f"  ✓ {lc_code} ({igbp_classes[lc_code]}): {result['n_pixels']:,} Pixel")

# === 2c: Trajektorien nach Landcover × Fire Count ===
print("\n2c. Trajektorien nach Landcover × Fire Count...")

lc_firecount_results = {}

for major_id in tqdm(major_classes_present, desc="LC × Fire Count"):
    lc_name = MAJOR_LC_NAMES[major_id]
    mask = (pre_fire_lc_major_id == major_id)
    lc_firecount_results[lc_name] = {}
    for fc in range(1, 5):
        result = calc_trajectory_by_fire_count(
            woody, burned, mask, years_woody, years_burned, nodata, fc, rel_years
        )
        if result is not None:
            lc_firecount_results[lc_name][fc] = result
            print(f"  ✓ {lc_name} × {fc} Fire(s): {result['n_pixels']:,} Pixel")

# === 2d: Trajektorien nach Landcover × Ecoregion (1 Fire) ===
print("\n2d. Trajektorien nach Landcover × Ecoregion (1 Fire)...")

lc_eco_results = {}

for eco_id in tqdm(unique_eco_ids, desc="LC × Ecoregion"):
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)
    lc_eco_results[eco_code] = {}
    for major_id in major_classes_present:
        lc_name = MAJOR_LC_NAMES[major_id]
        mask = (pre_fire_lc_major_id == major_id) & eco_mask
        result = calc_trajectory(woody, burned, mask, years_woody, years_burned, nodata, rel_years)
        if result is not None:
            lc_eco_results[eco_code][lc_name] = result

print(f"\n✓ Alle LC-Trajektorien berechnet!")



2. BERECHNE TRAJEKTORIEN NACH LANDCOVER
----------------------------------------------------------------------
Analysiere 6 Major LC Klassen: ['Forests', 'Shrublands', 'Open Woodland', 'Grasslands', 'Wetlands', 'Croplands']

2a. Trajektorien nach IGBP Major-Klasse (1 Fire Event)...


Major LC Klassen:  17%|█▋        | 1/6 [00:01<00:05,  1.16s/it]

  ✓ Forests: 157,654 Pixel


Major LC Klassen:  33%|███▎      | 2/6 [00:02<00:03,  1.02it/s]

  ✓ Shrublands: 12,943 Pixel


...

  ✓ Wetlands × 4 Fire(s): 998 Pixel
  ✓ Croplands × 1 Fire(s): 1,466,288 Pixel
  ✓ Croplands × 2 Fire(s): 570,652 Pixel
  ✓ Croplands × 3 Fire(s): 251,808 Pixel


LC × Fire Count: 100%|██████████| 6/6 [00:23<00:00,  3.90s/it]


  ✓ Croplands × 4 Fire(s): 116,801 Pixel

2d. Trajektorien nach Landcover × Ecoregion (1 Fire)...


LC × Ecoregion: 100%|██████████| 11/11 [00:46<00:00,  4.27s/it]


✓ Alle LC-Trajektorien berechnet!


In [18]:
# === SCHRITT 2e: TRAJEKTORIEN PRO ECOREGION (1 Fire) ===

print("\n2e. Trajektorien pro Ecoregion (1 Fire Event)...")

eco_results = {}

for eco_id in tqdm(unique_eco_ids, desc="Ecoregions (1 Fire)"):
    eco_code = eco_mapping[eco_id]['code']
    eco_name = eco_mapping[eco_id]['name']
    eco_color = eco_mapping[eco_id]['hex_color']
    eco_mask = (eco_raster == eco_id)
    
    result = calc_trajectory(
        woody, burned, eco_mask, years_woody, years_burned, nodata, rel_years
    )
    
    if result is not None:
        eco_results[eco_code] = {
            **result,
            'name': eco_name,
            'hex_color': eco_color,
            'eco_id': int(eco_id)
        }
        print(f"  ✓ {eco_code} ({eco_name}): {result['n_pixels']:,} Pixel")
    else:
        print(f"  ✗ {eco_code} ({eco_name}): zu wenige Pixel (<30)")

print(f"\n✓ {len(eco_results)} Ecoregion-Trajektorien berechnet")


2e. Trajektorien pro Ecoregion (1 Fire Event)...


Ecoregions (1 Fire):   9%|▉         | 1/11 [00:00<00:07,  1.35it/s]

  ✓ Anatolian (Anatolian Bio-geographical Region): 74,752 Pixel


Ecoregions (1 Fire):  18%|█▊        | 2/11 [00:01<00:06,  1.47it/s]

  ✓ BlackSea (Black Sea Bio-geographical Region): 8,331 Pixel


...

  ✓ Atlantic (Atlantic Bio-geographical Region): 37,725 Pixel


Ecoregions (1 Fire):  91%|█████████ | 10/11 [00:11<00:00,  1.06it/s]

  ✓ Outside (outside Europe): 165,297 Pixel


Ecoregions (1 Fire): 100%|██████████| 11/11 [00:12<00:00,  1.11s/it]

  ✓ Arctic (Arctic Bio-geographical Region): 915 Pixel

✓ 11 Ecoregion-Trajektorien berechnet


In [19]:
# === SCHRITT 2f: TRAJEKTORIEN PRO ECOREGION × FIRE COUNT ===

print("\n2f. Trajektorien pro Ecoregion × Fire Count...")

eco_firecount_results = {}

for eco_id in tqdm(unique_eco_ids, desc="Ecoregion × Fire Count"):
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)
    
    eco_firecount_results[eco_code] = {}
    
    for fc in range(1, 5):
        result = calc_trajectory_by_fire_count(
            woody, burned, eco_mask, years_woody, years_burned, nodata, fc, rel_years
        )
        if result is not None:
            eco_firecount_results[eco_code][fc] = result
            print(f"  ✓ {eco_code} × {fc} Fire(s): {result['n_pixels']:,} Pixel")

print(f"\n✓ Ecoregion × Fire Count Trajektorien berechnet")


2f. Trajektorien pro Ecoregion × Fire Count...


Ecoregion × Fire Count:   0%|          | 0/11 [00:00<?, ?it/s]

  ✓ Anatolian × 1 Fire(s): 74,752 Pixel
  ✓ Anatolian × 2 Fire(s): 27,558 Pixel
  ✓ Anatolian × 3 Fire(s): 14,773 Pixel


Ecoregion × Fire Count:   9%|▉         | 1/11 [00:02<00:25,  2.57s/it]

  ✓ Anatolian × 4 Fire(s): 9,543 Pixel
  ✓ BlackSea × 1 Fire(s): 8,331 Pixel
  ✓ BlackSea × 2 Fire(s): 2,991 Pixel
  ✓ BlackSea × 3 Fire(s): 1,651 Pixel


...

  ✓ Atlantic × 4 Fire(s): 454 Pixel
  ✓ Outside × 1 Fire(s): 165,297 Pixel
  ✓ Outside × 2 Fire(s): 62,643 Pixel
  ✓ Outside × 3 Fire(s): 37,404 Pixel


Ecoregion × Fire Count:  91%|█████████ | 10/11 [00:31<00:02,  2.85s/it]

  ✓ Outside × 4 Fire(s): 24,587 Pixel
  ✓ Arctic × 1 Fire(s): 915 Pixel


Ecoregion × Fire Count: 100%|██████████| 11/11 [00:33<00:00,  3.02s/it]


✓ Ecoregion × Fire Count Trajektorien berechnet


In [29]:
# === SCHRITT 2g: FIRE STATISTICS (vektorisiert) ===
# Fire Season Length, Fire Count, Burned Area – alles ohne Python-Loops über Pixel

print("\n2g. FIRE STATISTICS (vektorisiert)")
print("-" * 70)

import time
stats_start = time.time()

# === Lade MBA season_length und count Bänder ===
print("Lade MBA Fire Count & Season Length Bänder...")

with rasterio.open(burned_path) as src:
    # "count" = Index 1 in band_structure, "season_length" = Index 11
    count_band_indices  = [1 + (year - MBA_START_YEAR) * len(band_structure) + 1  for year in years_burned]
    season_band_indices = [1 + (year - MBA_START_YEAR) * len(band_structure) + 11 for year in years_burned]

    mba_count  = src.read(count_band_indices)     # (n_years, H, W)
    mba_season = src.read(season_band_indices)    # (n_years, H, W)

print(f"  ✓ MBA Count: {mba_count.shape}")
print(f"  ✓ MBA Season Length: {mba_season.shape}")

# === Pixelfläche ===
pixel_area_km2 = 0.25   # 500m × 500m
pixel_area_ha  = 25.0

# =========================================================
# BURNED AREA TOTAL – DataFrame ['Year', 'Burned_km2']
# → Plots E4, CSV 7
# =========================================================
print("\nBerechne Burned Area (gesamt)...")

rows_total = []
for yr_idx, year in enumerate(years_burned):
    n_burned = int(np.sum(mba_count[yr_idx] > 0))
    rows_total.append({'Year': year, 'Burned_km2': n_burned * pixel_area_km2})

burned_area_total = pd.DataFrame(rows_total)
print(f"  ✓ burned_area_total: {len(burned_area_total)} Jahre | "
      f"Ø {burned_area_total['Burned_km2'].mean():,.0f} km²/yr")

# =========================================================
# BURNED AREA PRO ECOREGION
# → DataFrame ['Year', 'Ecoregion', 'Burned_km2', 'Burned_Fraction']
# → Plots E5, CSV 8
# =========================================================
print("Berechne Burned Area pro Ecoregion...")

rows_eco = []
for yr_idx, year in enumerate(years_burned):
    burned_mask = (mba_count[yr_idx] > 0)
    for eco_id in unique_eco_ids:
        eco_code     = eco_mapping[eco_id]['code']
        eco_mask     = (eco_raster == eco_id)
        eco_total_px = int(np.sum(eco_mask))
        eco_burned_px = int(np.sum(burned_mask & eco_mask))
        rows_eco.append({
            'Year':            year,
            'Ecoregion':       eco_code,
            'Burned_km2':      eco_burned_px * pixel_area_km2,
            'Burned_Fraction': eco_burned_px / eco_total_px if eco_total_px > 0 else 0.0,
        })

burned_area_eco = pd.DataFrame(rows_eco)
print(f"  ✓ burned_area_eco: {len(burned_area_eco)} Eco×Jahr Einträge")

# =========================================================
# FIRE STATS SUMMARY PRO ECOREGION
# → DataFrame ['Ecoregion', 'Season_Length_*', 'Fire_Count_*']
# → Plots E6a, E7a, CSV 11
# =========================================================
print("Berechne Fire Stats Summary pro Ecoregion...")

stats_rows = []
for eco_id in unique_eco_ids:
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)

    s_vals = mba_season[:, eco_mask].flatten().astype(float)
    c_vals = mba_count[:, eco_mask].flatten().astype(float)
    s_vals = s_vals[(s_vals > 0) & np.isfinite(s_vals)]
    c_vals = c_vals[(c_vals > 0) & np.isfinite(c_vals)]

    if len(s_vals) > 0 and len(c_vals) > 0:
        stats_rows.append({
            'Ecoregion':          eco_code,
            'Season_Length_Min':  float(s_vals.min()),
            'Season_Length_Max':  float(s_vals.max()),
            'Season_Length_Mean': float(s_vals.mean()),
            'Season_Length_Std':  float(s_vals.std()),
            'Fire_Count_Min':     float(c_vals.min()),
            'Fire_Count_Max':     float(c_vals.max()),
            'Fire_Count_Mean':    float(c_vals.mean()),
            'Fire_Count_Std':     float(c_vals.std()),
        })

fire_stats_summary = pd.DataFrame(stats_rows)
# Alias für CSV 10 (fire_season_length_by_ecoregion.csv)
season_stats = fire_stats_summary.copy()

print(f"  ✓ fire_stats_summary: {len(fire_stats_summary)} Ecoregionen")
print(f"  {fire_stats_summary[['Ecoregion','Season_Length_Mean','Fire_Count_Mean']].to_string(index=False)}")

# =========================================================
# SEASON LENGTH × FIRE COUNT – dict für Plot E8
# → {fc_val: {'values': np.array, 'n': int, 'mean': float}}
# =========================================================
print("\nBerechne Season Length × Fire Count...")

count_flat  = mba_count.flatten().astype(float)
season_flat = mba_season.flatten().astype(float)
valid_mask  = (count_flat > 0) & (season_flat > 0) & np.isfinite(count_flat) & np.isfinite(season_flat)
count_valid  = count_flat[valid_mask].astype(int)
season_valid = season_flat[valid_mask]

season_by_fc = {}
for fc_val in np.unique(count_valid):
    fc_mask = (count_valid == fc_val)
    vals = season_valid[fc_mask]
    season_by_fc[int(fc_val)] = {
        'values': vals,
        'n':      int(len(vals)),
        'mean':   float(vals.mean()),
    }

print(f"  ✓ season_by_fc: Fire-Count-Werte = {sorted(season_by_fc.keys())}")
for k, v in sorted(season_by_fc.items()):
    print(f"    {k} Fire(s): n={v['n']:,}, mean season = {v['mean']:.2f} Monate")

# =========================================================
# FIRE COUNT VERTEILUNG PRO ECOREGION
# → DataFrame ['Ecoregion', 'Fire_Count', 'N_Pixels', 'Area_km2']
# → CSV 9
# =========================================================
print("\nBerechne Fire Count Verteilung pro Ecoregion...")

fc_rows = []
for eco_id in unique_eco_ids:
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)
    for fc_val in range(0, int(fire_counts.max()) + 1):
        n_px = int(np.sum((fire_counts == fc_val) & eco_mask))
        if n_px > 0:
            fc_rows.append({
                'Ecoregion':  eco_code,
                'Fire_Count': fc_val,
                'N_Pixels':   n_px,
                'Area_km2':   n_px * pixel_area_km2,
            })

fire_count_eco = pd.DataFrame(fc_rows)
print(f"  ✓ fire_count_eco: {len(fire_count_eco)} Zeilen")

# === ABSCHLUSS ===
stats_elapsed = time.time() - stats_start
print(f"\n✓ Schritt 2g abgeschlossen ({stats_elapsed:.1f}s)")
print(f"  burned_area_total:  {len(burned_area_total)} Jahre")
print(f"  burned_area_eco:    {len(burned_area_eco)} Eco×Jahr Einträge")
print(f"  fire_stats_summary: {len(fire_stats_summary)} Ecoregionen")
print(f"  season_by_fc:       {sorted(season_by_fc.keys())}")
print(f"  fire_count_eco:     {len(fire_count_eco)} Zeilen")



2g. FIRE STATISTICS (vektorisiert)
----------------------------------------------------------------------
Lade MBA Fire Count & Season Length Bänder...
  ✓ MBA Count: (25, 9660, 10483)
  ✓ MBA Season Length: (25, 9660, 10483)

Berechne Burned Area (gesamt)...
  ✓ burned_area_total: 25 Jahre | Ø 77,784 km²/yr
Berechne Burned Area pro Ecoregion...
  ✓ burned_area_eco: 275 Eco×Jahr Einträge
Berechne Fire Stats Summary pro Ecoregion...
  ✓ fire_stats_summary: 11 Ecoregionen
      Ecoregion  Season_Length_Mean  Fire_Count_Mean
    Anatolian            1.224774         1.048305
     BlackSea            1.025747         1.011661
      Steppic            1.041079         1.015806
  Continental            1.041948         1.014640
       Alpine            1.030403         1.009731
       Boreal            1.008240         1.006211
Mediterranean            1.022932         1.006066
    Pannonian            1.009350         1.004947
     Atlantic            1.010919         1.005570
      Outsid

In [13]:
# === SCHRITT 2h: RECOVERY-%-INDEX (loss-relativ, Fire Freq 1–4, bis +10yr) ===

print("\n2h. RECOVERY-%-INDEX (loss-relativ, Fire Freq 1–4, bis +10yr)")
print("-" * 70)

colors_fc = {1: '#1f77b4', 2: '#ff7f0e', 3: '#2ca02c', 4: '#d62728'}


def calc_trajectory_extended(woody, burned, eco_mask, years_woody, years_burned,
                              nodata, fire_count_value, fire_counts_arr, first_fire_idx_arr):
    """
    Berechnet Extended Trajectory für Pixel mit spezifischem Fire Count (−5 bis +10 Jahre).
    Nutzt vorberechnete fire_counts_arr und first_fire_idx_arr aus Step 1.
    """
    specific_fire_mask = (fire_counts_arr == fire_count_value) & eco_mask
    n_pixels = int(np.sum(specific_fire_mask))
    if n_pixels < 30:
        return None

    offset = years_burned[0] - years_woody[0]
    trajectory, trajectory_std = [], []

    for rel_year in range(-5, 11):
        woody_band = first_fire_idx_arr + rel_year + offset
        valid_mask = (woody_band >= 0) & (woody_band < len(years_woody)) & specific_fire_mask

        if np.sum(valid_mask) == 0:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)
            continue

        y_idx, x_idx = np.where(valid_mask)
        values = woody[woody_band[y_idx, x_idx], y_idx, x_idx]
        values = values[values != nodata]

        if len(values) > 0:
            trajectory.append(float(np.nanmean(values)))
            trajectory_std.append(float(np.nanstd(values)))
        else:
            trajectory.append(np.nan)
            trajectory_std.append(np.nan)

    return {'trajectory': trajectory, 'std': trajectory_std, 'n_pixels': n_pixels}


# --- Extended Trajectories berechnen ---
print("Berechne Extended Trajectories (10yr post-fire) pro Ecoregion × Fire Count...")
extended_trajectories = {}

for eco_id in tqdm(unique_eco_ids, desc="Extended Trajectories"):
    eco_code = eco_mapping[eco_id]['code']
    if eco_code not in eco_results:
        continue
    eco_mask = (eco_raster == eco_id)
    extended_trajectories[eco_code] = {}
    for fc_val in range(1, 5):
        result = calc_trajectory_extended(
            woody, burned, eco_mask, years_woody, years_burned,
            nodata, fc_val, fire_counts, first_fire_idx
        )
        if result is not None:
            extended_trajectories[eco_code][fc_val] = result

# --- Recovery-% berechnen ---
recovery_data_extended = []

for eco_code, fc_dict in extended_trajectories.items():
    for fc_val, traj_data in fc_dict.items():
        traj_arr = np.array(traj_data['trajectory'])   # 16 Werte: −5 bis +10
        pre_fire  = float(np.nanmean(traj_arr[:5]))    # Jahre −5 bis −1
        fire_year = float(traj_arr[5])                 # Jahr 0
        loss = pre_fire - fire_year

        for years_after in range(1, 11):
            rec_val = float(traj_arr[5 + years_after])
            rec_pct = ((rec_val - fire_year) / loss * 100) if loss > 0 else np.nan
            recovery_data_extended.append({
                'Ecoregion_Code':   eco_code,
                'Fire_Count':       fc_val,
                'N_Pixels':         traj_data['n_pixels'],
                'Years_After_Fire': years_after,
                'Pre_Fire_Cover':   pre_fire,
                'Fire_Year_Cover':  fire_year,
                'Recovery_Cover':   rec_val,
                'Woody_Loss':       loss,
                'Recovery_Percent': rec_pct
            })

recovery_df_extended = pd.DataFrame(recovery_data_extended)

# --- Plot: 2×2 Recovery % pro Fire Count ---
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for fc_idx, fc_val in enumerate([1, 2, 3, 4]):
    ax = axes[fc_idx // 2, fc_idx % 2]
    fc_data = recovery_df_extended[recovery_df_extended['Fire_Count'] == fc_val]
    all_vals = fc_data['Recovery_Percent'].dropna().values
    y_min = min(all_vals.min() - 10, -50) if len(all_vals) > 0 else -50
    y_max = max(all_vals.max() + 10, 120) if len(all_vals) > 0 else 120

    for eco_code in sorted(fc_data['Ecoregion_Code'].unique()):
        eco_data = fc_data[fc_data['Ecoregion_Code'] == eco_code]
        color = next(
            (eco_mapping[eid]['hex_color'] for eid in unique_eco_ids
             if eco_mapping[eid]['code'] == eco_code),
            '#808080'
        )
        ax.plot(eco_data['Years_After_Fire'], eco_data['Recovery_Percent'],
                marker='o', linewidth=2.5, markersize=8,
                label=f"{eco_code} (n={eco_data['N_Pixels'].iloc[0]:,})",
                color=color, alpha=0.85)

    ax.axhline(y=100, color='green', linestyle='--', linewidth=2, alpha=0.6, label='Full Recovery')
    ax.axhline(y=0,   color='gray',  linestyle=':',  linewidth=1.5, alpha=0.5, label='No Recovery')
    ax.set_xlabel('Years After Fire', fontsize=12)
    ax.set_ylabel('Recovery (%)', fontsize=12)
    ax.set_title(f'Recovery Rate: {fc_val} Fire Event(s)', fontsize=13, fontweight='bold')
    ax.legend(loc='best', fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0.5, 10.5)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(range(1, 11))

plt.suptitle('Woody Cover Recovery Rates by Fire Count (bis +10 Jahre)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "recovery_rates_by_fire_count_extended.png"), dpi=300, bbox_inches='tight')
plt.close()
print("✓ Plot: recovery_rates_by_fire_count_extended.png")

recovery_df_extended.to_csv(os.path.join(eco_csv_dir, "recovery_rates_by_fire_count_extended.csv"), index=False)
print("✓ CSV:  recovery_rates_by_fire_count_extended.csv")

print("\n" + "=" * 70)
for fc_val in [1, 2, 3, 4]:
    print(f"\n{fc_val} FIRE EVENT(S):")
    fc_data = recovery_df_extended[recovery_df_extended['Fire_Count'] == fc_val]
    for eco_code in sorted(fc_data['Ecoregion_Code'].unique()):
        eco_data = fc_data[fc_data['Ecoregion_Code'] == eco_code]
        r5  = eco_data[eco_data['Years_After_Fire'] == 5 ]['Recovery_Percent'].values
        r10 = eco_data[eco_data['Years_After_Fire'] == 10]['Recovery_Percent'].values
        r5_str  = f"{r5[0]:.1f}%"  if len(r5)  > 0 and not np.isnan(r5[0])  else "n/a"
        r10_str = f"{r10[0]:.1f}%" if len(r10) > 0 and not np.isnan(r10[0]) else "n/a"
        print(f"  {eco_code}: 5yr={r5_str}, 10yr={r10_str}")
print("=" * 70)



2h. RECOVERY-%-INDEX (loss-relativ, Fire Freq 1–4, bis +10yr)
----------------------------------------------------------------------
Berechne Extended Trajectories (10yr post-fire) pro Ecoregion × Fire Count...


Extended Trajectories: 100%|██████████| 11/11 [16:05<00:00, 87.78s/it]


✓ Plot: recovery_rates_by_fire_count_extended.png
✓ CSV:  recovery_rates_by_fire_count_extended.csv


1 FIRE EVENT(S):
  Alpine: 5yr=107.2%, 10yr=379.6%
  Anatolian: 5yr=n/a, 10yr=n/a
  Arctic: 5yr=-57.7%, 10yr=407.1%
  Atlantic: 5yr=36.1%, 10yr=70.2%
  BlackSea: 5yr=-48.7%, 10yr=-376.0%
  Boreal: 5yr=1625.3%, 10yr=2275.4%
  Continental: 5yr=-38.5%, 10yr=-39.8%
  Mediterranean: 5yr=11.6%, 10yr=33.8%
  Outside: 5yr=-110.8%, 10yr=-0.0%
  Pannonian: 5yr=n/a, 10yr=n/a
  Steppic: 5yr=-118.7%, 10yr=-131.1%

2 FIRE EVENT(S):
  Alpine: 5yr=14.8%, 10yr=169.1%
  Anatolian: 5yr=n/a, 10yr=n/a
  Atlantic: 5yr=130.6%, 10yr=195.8%
  BlackSea: 5yr=-56.5%, 10yr=-644.8%
  Boreal: 5yr=n/a, 10yr=n/a
  Continental: 5yr=1059.4%, 10yr=-8587.2%
  Mediterranean: 5yr=43.6%, 10yr=109.7%
  Outside: 5yr=-188.5%, 10yr=-128.9%
  Pannonian: 5yr=487.2%, 10yr=980.4%
  Steppic: 5yr=-189.4%, 10yr=-212.9%

3 FIRE EVENT(S):
  Alpine: 5yr=71.4%, 10yr=436.8%
  Anatolian: 5yr=n/a, 10yr=n/a
  Atlantic: 5yr=98.7%, 10yr=232.9%
 

In [21]:
# === SCHRITT 2i: HEATMAP Ecoregion × Fire Count (Fläche km²/%) + Ecoregion × LC ===

print("\n2i. HEATMAP ECOREGION × FIRE COUNT (Fläche km²/%) + ECOREGION × LC")
print("-" * 70)

pixel_area_km2 = 0.25   # 500m × 500m
pixel_area_ha  = 25.0

# Eco-Gesamtflächen (aus eco_raster)
eco_total_areas_km2 = {
    eco_mapping[eco_id]['code']: float(np.sum(eco_raster == eco_id)) * pixel_area_km2
    for eco_id in unique_eco_ids
}

# --- Summary Statistics: Ecoregion × Fire Count ---
summary_stats = []

for eco_id in unique_eco_ids:
    eco_code = eco_mapping[eco_id]['code']
    eco_name = eco_mapping[eco_id]['name']
    if eco_code not in eco_firecount_results:
        continue
    eco_total_area = eco_total_areas_km2[eco_code]

    for fc_val, traj_data in eco_firecount_results[eco_code].items():
        traj_arr     = np.array(traj_data['trajectory'])   # 16 Werte: −5 bis +10
        pre_fire     = float(np.nanmean(traj_arr[:5]))     # Jahre −5 bis −1
        fire_year    = float(traj_arr[5])                  # Jahr 0
        rec_5yr      = float(traj_arr[10])                 # Jahr +5
        rec_10yr     = float(traj_arr[15]) if len(traj_arr) > 15 else np.nan  # Jahr +10
        loss         = pre_fire - fire_year
        rec_pct_5yr  = ((rec_5yr  - fire_year) / loss * 100) if loss > 0 else np.nan
        rec_pct_10yr = ((rec_10yr - fire_year) / loss * 100) if (loss > 0 and not np.isnan(rec_10yr)) else np.nan
        area_km2     = traj_data['n_pixels'] * pixel_area_km2

        summary_stats.append({
            'Ecoregion_Code':        eco_code,
            'Ecoregion_Name':        eco_name,
            'Fire_Count':            fc_val,
            'N_Pixels':              traj_data['n_pixels'],
            'Area_km2':              area_km2,
            'Area_ha':               traj_data['n_pixels'] * pixel_area_ha,
            'Percent_of_Ecoregion':  (area_km2 / eco_total_area * 100) if eco_total_area > 0 else 0,
            'Ecoregion_Total_km2':   eco_total_area,
            'Pre_Fire_Cover':        pre_fire,
            'Post_Fire_Cover':       fire_year,
            'Recovery_5yr_Cover':    rec_5yr,
            'Woody_Loss':            loss,
            'Recovery_Percent_5yr':  rec_pct_5yr,
            'Recovery_Percent_10yr': rec_pct_10yr
        })

summary_df_eco = pd.DataFrame(summary_stats)
eco_codes_summary = sorted(summary_df_eco['Ecoregion_Code'].unique())

summary_csv = os.path.join(lcxeco_csv_dir, "summary_statistics_by_fire_count.csv")
summary_df_eco.to_csv(summary_csv, index=False)
print(f"✓ CSV: {summary_csv}")

# --- Heatmap 1: Ecoregion × Fire Count (4 Metriken) ---
print("Erstelle Heatmap Ecoregion × Fire Count...")

metrics_fc  = ['Area_km2', 'Percent_of_Ecoregion', 'Woody_Loss', 'Recovery_Percent_5yr']
titles_fc   = ['Betroffene Fläche (km²)', 'Betroffene Fläche (% der Region)',
               'Woody Cover Verlust (%)', 'Recovery nach 5 Jahren (%)']
cmaps_fc    = ['YlOrRd', 'YlOrRd', 'Reds', 'RdYlGn']

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

for idx, (metric, title, cmap) in enumerate(zip(metrics_fc, titles_fc, cmaps_fc)):
    ax = axes[idx // 2, idx % 2]
    pivot = summary_df_eco.pivot(index='Fire_Count', columns='Ecoregion_Code', values=metric)
    im = ax.imshow(pivot.values, aspect='auto', cmap=cmap)

    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticklabels([f'{int(fc_val)} Fire(s)' for fc_val in pivot.index])

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                text_color = 'white' if val > np.nanmean(pivot.values) else 'black'
                txt = f'{val:.1f}' if metric == 'Area_km2' else f'{val:.1f}%'
                ax.text(j, i, txt, ha='center', va='center',
                        color=text_color, fontsize=10, fontweight='bold')

    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('Ecoregion', fontsize=11)
    ax.set_ylabel('Fire Count', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(labelsize=9)

plt.suptitle('Summary Heatmap: Ecoregion × Fire Count', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(os.path.join(lcxeco_plots_dir, "summary_heatmap_ecoregion_firecount.png"), dpi=300, bbox_inches='tight')
plt.close()
print("✓ Heatmap: summary_heatmap_ecoregion_firecount.png")

# --- Heatmap 2: Ecoregion × Pre-Fire LC (3 Metriken) ---
print("Erstelle Heatmap Ecoregion × Pre-Fire LC...")

lc_classes_avail = sorted({lc for eco in lc_eco_results for lc in lc_eco_results[eco]})
eco_codes_lc     = sorted(lc_eco_results.keys())

metrics_lc = ['loss', 'recovery_5yr', 'n_pixels']
titles_lc  = ['Sofort-Verlust (%)', 'Recovery 5yr (%)', 'Anzahl Pixel (n)']
cmaps_lc   = ['Reds', 'RdYlGn', 'Blues']

fig, axes_lc = plt.subplots(1, 3, figsize=(22, max(6, len(eco_codes_lc) * 0.8 + 2)))

for idx, (metric, title, cmap) in enumerate(zip(metrics_lc, titles_lc, cmaps_lc)):
    ax = axes_lc[idx]
    mat = np.full((len(eco_codes_lc), len(lc_classes_avail)), np.nan)

    for i, eco_code in enumerate(eco_codes_lc):
        for j, lc_name in enumerate(lc_classes_avail):
            if lc_name not in lc_eco_results.get(eco_code, {}):
                continue
            td       = lc_eco_results[eco_code][lc_name]
            ta       = np.array(td['trajectory'])
            pre      = float(np.nanmean(ta[:5]))
            fire     = float(ta[5])
            rec      = float(ta[10])
            loss_val = pre - fire
            rec_5pct = ((rec - fire) / loss_val * 100) if loss_val > 0 else np.nan

            if metric == 'loss':
                mat[i, j] = loss_val
            elif metric == 'recovery_5yr':
                mat[i, j] = rec_5pct
            else:
                mat[i, j] = td['n_pixels']

    im = ax.imshow(np.ma.masked_invalid(mat), aspect='auto', cmap=cmap)
    ax.set_xticks(np.arange(len(lc_classes_avail)))
    ax.set_yticks(np.arange(len(eco_codes_lc)))
    ax.set_xticklabels(lc_classes_avail, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(eco_codes_lc, fontsize=9)

    valid = mat[~np.isnan(mat)]
    mean_val = valid.mean() if len(valid) > 0 else 0
    for i in range(len(eco_codes_lc)):
        for j in range(len(lc_classes_avail)):
            val = mat[i, j]
            if not np.isnan(val):
                text_color = 'white' if val > mean_val else 'black'
                txt = f'{val:.0f}' if metric == 'n_pixels' else f'{val:.1f}%'
                ax.text(j, i, txt, ha='center', va='center',
                        color=text_color, fontsize=8, fontweight='bold')

    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Pre-Fire Land Cover', fontsize=10)
    ax.set_ylabel('Ecoregion', fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(labelsize=9)

plt.suptitle('Ecoregion × Pre-Fire Land Cover: Loss & Recovery (Single Fire Events)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(lcxeco_plots_dir, "summary_heatmap_ecoregion_lc.png"), dpi=300, bbox_inches='tight')
plt.close()
print("✓ Heatmap: summary_heatmap_ecoregion_lc.png")

print("\n" + "=" * 70)
print("✓✓✓ SCHRITT 2i ABGESCHLOSSEN ✓✓✓")
print("=" * 70)
print(f"  CSV:    {summary_csv}")
print(f"  Plots:  summary_heatmap_ecoregion_firecount.png")
print(f"          summary_heatmap_ecoregion_lc.png")
print("=" * 70)



2i. HEATMAP ECOREGION × FIRE COUNT (Fläche km²/%) + ECOREGION × LC
----------------------------------------------------------------------
✓ CSV: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\LC_x_Ecoregion\csv\summary_statistics_by_fire_count.csv
Erstelle Heatmap Ecoregion × Fire Count...
✓ Heatmap: summary_heatmap_ecoregion_firecount.png
Erstelle Heatmap Ecoregion × Pre-Fire LC...
✓ Heatmap: summary_heatmap_ecoregion_lc.png

✓✓✓ SCHRITT 2i ABGESCHLOSSEN ✓✓✓
  CSV:    \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\LC_x_Ecoregion\csv\summary_statistics_by_fire_count.csv
  Plots:  summary_heatmap_ecoregion_firecount.png
          summary_heatmap_ecoregion_lc.png


In [23]:
# === SCHRITT 3: VISUALISIERUNGEN ===

print("\n3. ERSTELLE VISUALISIERUNGEN")
print("-" * 70)

lc_plots_dir = os.path.join(workDir, "_Runs", "09_Results", "Landcover", "plots")
lcxeco_plots_dir = os.path.join(workDir, "_Runs", "09_Results", "LC_x_Ecoregion", "plots")
os.makedirs(lc_plots_dir, exist_ok=True)
os.makedirs(lcxeco_plots_dir, exist_ok=True)

# Farben für Landcover-Typen
LC_COLORS = {
    'Forests':       '#228B22',
    'Shrublands':    '#8B4513',
    'Open Woodland': '#DAA520',
    'Grasslands':    '#90EE90',
    'Wetlands':      '#4682B4',
    'Croplands':     '#FFD700',
}

FIRE_COUNT_COLORS = {
    1: '#1f77b4',
    2: '#ff7f0e',
    3: '#2ca02c',
    4: '#d62728'
}

# =====================================================
# PLOT 1: Alle Landcover-Trajektorien (1 Fire) combined
# =====================================================
print("\nPlot 1: Alle LC-Trajektorien (1 Fire)...")

fig, ax = plt.subplots(figsize=(14, 8))

for lc_class, data in sorted(lc_major_results.items()):
    color = LC_COLORS.get(lc_class, '#808080')
    traj = np.array(data['trajectory'])
    std = np.array(data['std'])
    
    ax.plot(rel_years, traj, marker='o', linewidth=2.5, 
            label=f"{lc_class} (n={data['n_pixels']:,})",
            color=color, alpha=0.9)
    
    lower = np.maximum(traj - std, 0)
    upper = np.minimum(traj + std, 100)
    ax.fill_between(rel_years, lower, upper, alpha=0.15, color=color)

ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Fire Year')
ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)

ax.set_xlabel('Years Relative to Fire Event', fontsize=12)
ax.set_ylabel('Woody Cover (%)', fontsize=12)
ax.set_title('Post-Fire Woody Cover Trajectories by Pre-Fire Land Cover Type\n(Single Fire Events Only)', 
             fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(-5.5, 10.5)

ax.annotate('Pre-Fire', xy=(-3, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkgreen', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))
ax.annotate('Fire\nYear', xy=(0.5, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkred', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='mistyrose', alpha=0.5))
ax.annotate('Recovery Period', xy=(5.5, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkblue', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.tight_layout()
plot1_path = os.path.join(lc_plots_dir, "lc_trajectories_1fire_all.png")
plt.savefig(plot1_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {plot1_path}")

# =====================================================
# PLOT 2: Einzelne Subplots pro Landcover-Typ (1 Fire)
# =====================================================
print("\nPlot 2: Subplots pro LC-Typ (1 Fire)...")

n_lc = len(lc_major_results)
n_cols = 3
n_rows = int(np.ceil(n_lc / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes = axes.flatten()

for i, (lc_class, data) in enumerate(sorted(lc_major_results.items())):
    ax = axes[i]
    color = LC_COLORS.get(lc_class, '#808080')
    traj = np.array(data['trajectory'])
    std = np.array(data['std'])
    
    lower = np.maximum(traj - std, 0)
    upper = np.minimum(traj + std, 100)
    
    y_min = max(0, np.nanmin(lower) - 5)
    y_max = min(100, np.nanmax(upper) + 5)
    
    ax.plot(rel_years, traj, marker='o', linewidth=2.5, color=color)
    ax.fill_between(rel_years, lower, upper, alpha=0.3, color=color)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)
    
    ax.set_ylim(y_min, y_max)
    ax.set_title(f"{lc_class}\n(n={data['n_pixels']:,} Events)", fontsize=11, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire', fontsize=9)
    ax.set_ylabel('Woody Cover (%)', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)

for i in range(n_lc, len(axes)):
    axes[i].axis('off')

plt.suptitle('Woody Cover Trajectories by Pre-Fire Land Cover (Single Fire Events)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plot2_path = os.path.join(lc_plots_dir, "lc_trajectories_1fire_panel.png")
plt.savefig(plot2_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {plot2_path}")

# =====================================================
# PLOT 3: Trajektorien nach Landcover × Fire Count
# =====================================================
print("\nPlot 3: LC × Fire Count Trajektorien...")

n_lc_fc = len([lc for lc in lc_firecount_results if lc_firecount_results[lc]])
n_cols = 2
n_rows = int(np.ceil(n_lc_fc / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 6 * n_rows))
axes = axes.flatten()

plot_idx = 0
for lc_class in sorted(lc_firecount_results.keys()):
    fc_data = lc_firecount_results[lc_class]
    if not fc_data:
        continue
    
    ax = axes[plot_idx]
    
    for fc, data in sorted(fc_data.items()):
        traj = np.array(data['trajectory'])
        std = np.array(data['std'])
        
        ax.plot(rel_years, traj, marker='o', linewidth=2.5,
                color=FIRE_COUNT_COLORS[fc],
                label=f'{fc} Fire(s) (n={data["n_pixels"]:,})',
                alpha=0.8)
        
        lower = np.maximum(traj - std, 0)
        upper = np.minimum(traj + std, 100)
        ax.fill_between(rel_years, lower, upper, alpha=0.15, color=FIRE_COUNT_COLORS[fc])
    
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax.set_title(f"{lc_class}", fontsize=12, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire', fontsize=10)
    ax.set_ylabel('Woody Cover (%)', fontsize=10)
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5.5, 10.5)
    
    lc_color = LC_COLORS.get(lc_class, '#808080')
    for spine in ax.spines.values():
        spine.set_edgecolor(lc_color)
        spine.set_linewidth(2)
    
    plot_idx += 1

for i in range(plot_idx, len(axes)):
    axes[i].axis('off')

plt.suptitle('Woody Cover Trajectories by Land Cover × Fire Count', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plot3_path = os.path.join(lc_plots_dir, "lc_trajectories_by_fire_count.png")
plt.savefig(plot3_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {plot3_path}")

# =====================================================
# PLOT 4: Heatmap Landcover × Ecoregion
# =====================================================
print("\nPlot 4: Heatmap LC × Ecoregion...")

heatmap_data_loss = {}
heatmap_data_recovery = {}
heatmap_data_npixels = {}

for eco_code, lc_data in lc_eco_results.items():
    heatmap_data_loss[eco_code] = {}
    heatmap_data_recovery[eco_code] = {}
    heatmap_data_npixels[eco_code] = {}
    
    for lc_class, data in lc_data.items():
        traj = np.array(data['trajectory'])
        pre_fire = np.nanmean(traj[:5])
        fire_year = traj[5]
        recovery_5yr = traj[10] if len(traj) > 10 else traj[-1]
        loss = pre_fire - fire_year if not np.isnan(fire_year) else np.nan
        rec_pct = ((recovery_5yr - fire_year) / loss * 100) if loss > 0 else np.nan
        
        heatmap_data_loss[eco_code][lc_class] = loss
        heatmap_data_recovery[eco_code][lc_class] = rec_pct
        heatmap_data_npixels[eco_code][lc_class] = data['n_pixels']

df_loss = pd.DataFrame(heatmap_data_loss).T
df_recovery_hm = pd.DataFrame(heatmap_data_recovery).T
df_npixels = pd.DataFrame(heatmap_data_npixels).T

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

ax = axes[0]
sns.heatmap(df_loss, cmap='Reds', annot=True, fmt='.1f', ax=ax,
            cbar_kws={'label': 'Woody Cover Loss (%)'})
ax.set_title('Immediate Woody Cover Loss\n(Pre-Fire − Fire Year)', fontsize=12, fontweight='bold')
ax.set_xlabel('Land Cover Type')
ax.set_ylabel('Ecoregion')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

ax = axes[1]
sns.heatmap(df_recovery_hm, cmap='RdYlGn', annot=True, fmt='.0f', ax=ax,
            center=100, cbar_kws={'label': 'Recovery after 5 Years (%)'})
ax.set_title('Recovery after 5 Years (%)\n(100% = Full Recovery)', fontsize=12, fontweight='bold')
ax.set_xlabel('Land Cover Type')
ax.set_ylabel('Ecoregion')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

ax = axes[2]
sns.heatmap(df_npixels, cmap='Blues', annot=True, fmt='.0f', ax=ax,
            cbar_kws={'label': 'Number of Pixels'})
ax.set_title('Sample Size\n(Pixels with Single Fire)', fontsize=12, fontweight='bold')
ax.set_xlabel('Land Cover Type')
ax.set_ylabel('Ecoregion')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle('Land Cover × Ecoregion: Fire Impact & Recovery Summary', 
             fontsize=15, fontweight='bold')
plt.tight_layout()
plot4_path = os.path.join(lcxeco_plots_dir, "lc_ecoregion_heatmap.png")
plt.savefig(plot4_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {plot4_path}")

# =====================================================
# PLOT 5: Trajektorien pro Ecoregion × Landcover
# =====================================================
print("\nPlot 5: Trajektorien pro Ecoregion × Landcover...")

for eco_code, lc_data in sorted(lc_eco_results.items()):
    if not lc_data:
        continue
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for lc_class, data in sorted(lc_data.items()):
        color = LC_COLORS.get(lc_class, '#808080')
        traj = np.array(data['trajectory'])
        
        ax.plot(rel_years, traj, marker='o', linewidth=2.5,
                label=f"{lc_class} (n={data['n_pixels']:,})",
                color=color, alpha=0.9)
    
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)
    
    eco_name = eco_mapping.get(
        [k for k, v in eco_mapping.items() if v['code'] == eco_code][0], {}
    ).get('name', eco_code)
    
    ax.set_title(f'{eco_code} – {eco_name}\nWoody Cover Trajectories by Pre-Fire Land Cover', 
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire Event', fontsize=12)
    ax.set_ylabel('Woody Cover (%)', fontsize=12)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5.5, 10.5)
    
    plt.tight_layout()
    plot5_path = os.path.join(lcxeco_plots_dir, f"lc_trajectories_{eco_code}.png")
    plt.savefig(plot5_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ {plot5_path}")



3. ERSTELLE VISUALISIERUNGEN
----------------------------------------------------------------------

Plot 1: Alle LC-Trajektorien (1 Fire)...
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Landcover\plots\lc_trajectories_1fire_all.png

Plot 2: Subplots pro LC-Typ (1 Fire)...
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Landcover\plots\lc_trajectories_1fire_panel.png

Plot 3: LC × Fire Count Trajektorien...
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Landcover\plots\lc_trajectories_by_fire_count.png

Plot 4: Heatmap LC × Ecoregion...
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\LC_x_Ecoregion\plots\lc_ecoregion_heatmap.png

Plot 5: Trajektorien pro Ecoregion × Landcover...
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\LC_x_Ecoregion\plots\lc_trajectories_Alpine.png
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\LC_x_Ecoregion\plots\lc_traject

In [24]:
# === SCHRITT 3b: STATISTISCHE TESTS – LANDCOVER-VERGLEICHE ===
# Kruskal-Wallis H-Test + Dunn's Post-hoc auf Pixel-Ebene
# Testet: Unterscheiden sich Woody-Cover-Metriken signifikant zwischen LC-Typen?

print("\n3b. STATISTISCHE TESTS – LANDCOVER-VERGLEICHE")
print("-" * 70)

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

from scipy import stats

try:
    import scikit_posthocs as sp
    HAS_POSTHOC = True
    print("✓ scikit-posthocs verfügbar")
except ImportError:
    HAS_POSTHOC = False
    print("⚠️  scikit-posthocs nicht installiert – nur Kruskal-Wallis (kein Dunn Post-hoc)")
    print("    Installation: pip install scikit-posthocs")

def extract_pixel_metrics_lc(woody, burned, fire_counts, first_fire_idx, mask,
                              years_woody, years_burned, nodata, max_samples=50000):
    """Extrahiert Pixel-Level Metriken für eine gegebene Maske (nur 1 Brand)."""
    single_fire_mask = (fire_counts == 1) & mask
    y_all, x_all = np.where(single_fire_mask)
    n_pixels = len(y_all)
    if n_pixels == 0:
        return None
    if n_pixels > max_samples:
        rng = np.random.default_rng(42)
        idx = rng.choice(n_pixels, max_samples, replace=False)
        y_all = y_all[idx]
        x_all = x_all[idx]
        n_pixels = max_samples
    offset = years_burned[0] - years_woody[0]
    fire_idx_subset = first_fire_idx[y_all, x_all]
    rel_years_extract = list(range(-5, 11))
    pixel_values = np.full((n_pixels, len(rel_years_extract)), np.nan)
    for j, rel_year in enumerate(rel_years_extract):
        woody_band_idx = fire_idx_subset + rel_year + offset - 1
        valid = (woody_band_idx >= 0) & (woody_band_idx < len(years_woody))
        if np.sum(valid) > 0:
            vals = woody[woody_band_idx[valid], y_all[valid], x_all[valid]].astype(float)
            vals[vals == nodata] = np.nan
            pixel_values[valid, j] = vals
    pre_fire_vals = pixel_values[:, 0:5]
    fire_year_val = pixel_values[:, 5]
    pre_fire_m1 = pixel_values[:, 4]
    recovery_5yr_val = pixel_values[:, 10]
    pre_fire_mean = np.nanmean(pre_fire_vals, axis=1)
    pre_fire_trend = np.full(n_pixels, np.nan)
    x_trend = np.arange(5)
    for i in range(n_pixels):
        vals_i = pre_fire_vals[i, :]
        valid_i = ~np.isnan(vals_i)
        if np.sum(valid_i) >= 3:
            slope, _, _, _, _ = stats.linregress(x_trend[valid_i], vals_i[valid_i])
            pre_fire_trend[i] = slope
    fire_loss = pre_fire_m1 - fire_year_val
    fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
    recovery_5yr_pct = np.where(fire_loss > 0,
                                (recovery_5yr_val - fire_year_val) / fire_loss * 100, np.nan)
    return {
        'pre_fire_mean': pre_fire_mean, 'pre_fire_trend': pre_fire_trend,
        'fire_loss': fire_loss, 'fire_loss_pct': fire_loss_pct,
        'recovery_5yr_pct': recovery_5yr_pct, 'woody_at_fire': fire_year_val,
        'n_pixels_total': int(np.sum(single_fire_mask)), 'n_pixels_sampled': n_pixels
    }

def kruskal_wallis_with_effect_size(groups, group_names, metric_name):
    """Kruskal-Wallis H-Test mit Eta-squared Effektstärke."""
    clean_groups, clean_names = [], []
    for g, name in zip(groups, group_names):
        valid = g[~np.isnan(g)]
        if len(valid) >= 5:
            clean_groups.append(valid)
            clean_names.append(name)
    if len(clean_groups) < 2:
        return None
    H_stat, p_value = stats.kruskal(*clean_groups)
    k = len(clean_groups)
    n_total = sum(len(g) for g in clean_groups)
    eta_squared = max(0, (H_stat - k + 1) / (n_total - k))
    effect_interp = ("negligible" if eta_squared < 0.01 else
                     "small" if eta_squared < 0.06 else
                     "medium" if eta_squared < 0.14 else "large")
    return {
        'metric': metric_name, 'H_statistic': H_stat, 'p_value': p_value,
        'eta_squared': eta_squared, 'effect_size': effect_interp, 'k_groups': k,
        'n_total': n_total, 'medians': {n: np.nanmedian(g) for n, g in zip(clean_names, clean_groups)},
        'n_per_group': {n: len(g) for n, g in zip(clean_names, clean_groups)},
        'significant': p_value < 0.05, 'groups': clean_groups, 'group_names': clean_names
    }

METRICS_TO_TEST = ['pre_fire_mean', 'pre_fire_trend', 'fire_loss', 'fire_loss_pct', 'recovery_5yr_pct']
METRIC_LABELS = {
    'pre_fire_mean': 'Pre-Fire Woody Cover (Mean, %)',
    'pre_fire_trend': 'Pre-Fire Trend (Slope, %/yr)',
    'fire_loss': 'Absolute Fire Loss (%)',
    'fire_loss_pct': 'Relative Fire Loss (%)',
    'recovery_5yr_pct': '5-Year Recovery (% of Loss)'
}
MAX_SAMPLES = 50000

# === TEST A: Vergleich zwischen Major LC-Klassen (Europa-weit) ===
print("\n" + "=" * 70)
print("TEST A: Kruskal-Wallis – Major Landcover Klassen (Europa-weit)")
print("=" * 70)
print("\nExtrahiere Pixel-Level Metriken pro LC-Klasse...")

lc_pixel_metrics = {}
for major_id in tqdm(major_classes_present, desc="LC Klassen"):
    lc_name = MAJOR_LC_NAMES[major_id]
    mask = (pre_fire_lc_major_id == major_id)
    metrics = extract_pixel_metrics_lc(woody, burned, fire_counts, first_fire_idx, mask,
                                       years_woody, years_burned, nodata, max_samples=MAX_SAMPLES)
    if metrics is not None:
        lc_pixel_metrics[lc_name] = metrics
        print(f"  ✓ {lc_name}: {metrics['n_pixels_total']:,} Pixel "
              f"(sampled: {metrics['n_pixels_sampled']:,})")

lc_names_ordered = sorted(lc_pixel_metrics.keys())
kw_results_lc = {}
for metric in METRICS_TO_TEST:
    groups = [lc_pixel_metrics[lc][metric] for lc in lc_names_ordered]
    result = kruskal_wallis_with_effect_size(groups, lc_names_ordered, metric)
    if result is not None:
        kw_results_lc[metric] = result
        sig_str = "***" if result['p_value'] < 0.001 else ("**" if result['p_value'] < 0.01 else ("*" if result['p_value'] < 0.05 else "n.s."))
        print(f"\n  {METRIC_LABELS[metric]}:")
        print(f"    H = {result['H_statistic']:.1f}, p = {result['p_value']:.2e} {sig_str}")
        print(f"    η² = {result['eta_squared']:.4f} ({result['effect_size']})")

dunn_results_lc = {}
if HAS_POSTHOC:
    print("\n  Dunn's Post-hoc Tests (Bonferroni):")
    for metric in METRICS_TO_TEST:
        if metric not in kw_results_lc or not kw_results_lc[metric]['significant']:
            continue
        result = kw_results_lc[metric]
        all_values, all_labels = [], []
        for g, name in zip(result['groups'], result['group_names']):
            all_values.extend(g.tolist())
            all_labels.extend([name] * len(g))
        dunn_input_df = pd.DataFrame({'value': all_values, 'group': all_labels})
        dunn_df = sp.posthoc_dunn(a=dunn_input_df, val_col='value',
                                   group_col='group', p_adjust='bonferroni')
        dunn_results_lc[metric] = dunn_df
        n_pairs = dunn_df.shape[0] * (dunn_df.shape[0] - 1) // 2
        sig_pairs = np.sum(dunn_df.values[np.triu_indices_from(dunn_df.values, k=1)] < 0.05)
        print(f"    {metric}: {sig_pairs}/{n_pairs} signifikante Paare")

# === TEST B: LC-Vergleich INNERHALB jeder Ecoregion ===
print("\n" + "=" * 70)
print("TEST B: Kruskal-Wallis – LC-Klassen innerhalb jeder Ecoregion")
print("=" * 70)
kw_results_lc_per_eco = {}
for eco_id in tqdm(unique_eco_ids, desc="Ecoregions"):
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)
    eco_lc_metrics = {}
    for major_id in major_classes_present:
        lc_name = MAJOR_LC_NAMES[major_id]
        mask = (pre_fire_lc_major_id == major_id) & eco_mask
        metrics = extract_pixel_metrics_lc(woody, burned, fire_counts, first_fire_idx, mask,
                                           years_woody, years_burned, nodata, max_samples=20000)
        if metrics is not None:
            eco_lc_metrics[lc_name] = metrics
    if len(eco_lc_metrics) < 2:
        continue
    kw_results_lc_per_eco[eco_code] = {}
    lc_names_eco = sorted(eco_lc_metrics.keys())
    for metric in METRICS_TO_TEST:
        groups = [eco_lc_metrics[lc][metric] for lc in lc_names_eco if lc in eco_lc_metrics]
        names = [lc for lc in lc_names_eco if lc in eco_lc_metrics]
        result = kruskal_wallis_with_effect_size(groups, names, metric)
        if result is not None:
            kw_results_lc_per_eco[eco_code][metric] = result

# === VISUALISIERUNGEN ===
print("\nErstelle Visualisierungen...")

# Boxplots
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()
for i, metric in enumerate(METRICS_TO_TEST):
    ax = axes[i]
    box_data, box_labels, box_colors = [], [], []
    for lc_name in lc_names_ordered:
        if lc_name not in lc_pixel_metrics:
            continue
        vals = lc_pixel_metrics[lc_name][metric]
        valid = vals[~np.isnan(vals)]
        if len(valid) > 0:
            box_data.append(valid)
            box_labels.append(lc_name)
            box_colors.append(LC_COLORS.get(lc_name, '#808080'))
    bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True, showfliers=False, widths=0.6)
    for j, color in enumerate(box_colors):
        bp['boxes'][j].set_facecolor(color)
        bp['boxes'][j].set_alpha(0.6)
    ax.set_title(METRIC_LABELS[metric], fontsize=11, fontweight='bold')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    if metric in kw_results_lc:
        kw = kw_results_lc[metric]
        sig_str = "***" if kw['p_value'] < 0.001 else ("**" if kw['p_value'] < 0.01 else ("*" if kw['p_value'] < 0.05 else "n.s."))
        ax.text(0.02, 0.98, f"KW: H={kw['H_statistic']:.0f}, η²={kw['eta_squared']:.3f} {sig_str}",
                transform=ax.transAxes, fontsize=8, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
axes[-1].axis('off')
plt.suptitle('Pixel-Level Metric Distributions by Pre-Fire Land Cover (Single Fire Events)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
boxplot_lc_path = os.path.join(lc_plots_dir, "statistical_tests_landcover_boxplots.png")
plt.savefig(boxplot_lc_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ {boxplot_lc_path}")

# Dunn Heatmaps
if HAS_POSTHOC and dunn_results_lc:
    n_metrics_sig = len(dunn_results_lc)
    fig, axes = plt.subplots(1, n_metrics_sig, figsize=(7 * n_metrics_sig, 6))
    if n_metrics_sig == 1:
        axes = [axes]
    plot_i = 0
    for metric in METRICS_TO_TEST:
        if metric not in dunn_results_lc:
            continue
        ax = axes[plot_i]
        dunn_df = dunn_results_lc[metric]
        log_p = np.clip(-np.log10(dunn_df.values.astype(float)), 0, 50)
        annot = dunn_df.copy()
        for r in annot.index:
            for c in annot.columns:
                p = dunn_df.loc[r, c]
                annot.loc[r, c] = ("" if r == c else "***" if p < 0.001 else "**" if p < 0.01
                                   else "*" if p < 0.05 else "n.s.")
        sns.heatmap(pd.DataFrame(log_p, index=dunn_df.index, columns=dunn_df.columns),
                    cmap='RdYlGn_r', annot=annot.values, fmt='', ax=ax,
                    square=True, linewidths=0.5, cbar_kws={'label': '-log₁₀(p)'}, vmin=0, vmax=10)
        ax.set_title(f'{METRIC_LABELS[metric]}\n(Dunn Post-hoc, Bonferroni)', fontsize=10, fontweight='bold')
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
        plot_i += 1
    plt.suptitle("Pairwise Land Cover Comparisons – Dunn's Post-hoc Test", fontsize=14, fontweight='bold')
    plt.tight_layout()
    heatmap_lc_path = os.path.join(lc_plots_dir, "statistical_tests_landcover_dunn_heatmaps.png")
    plt.savefig(heatmap_lc_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ {heatmap_lc_path}")

# Effektstärken-Heatmap
if kw_results_lc_per_eco:
    eco_codes_sorted = sorted(kw_results_lc_per_eco.keys())
    eta_matrix = pd.DataFrame(index=eco_codes_sorted, columns=METRICS_TO_TEST, dtype=float)
    sig_matrix = pd.DataFrame(index=eco_codes_sorted, columns=METRICS_TO_TEST, dtype=str)
    for eco_code in eco_codes_sorted:
        for metric in METRICS_TO_TEST:
            if metric in kw_results_lc_per_eco[eco_code]:
                r = kw_results_lc_per_eco[eco_code][metric]
                eta_matrix.loc[eco_code, metric] = r['eta_squared']
                sig_str = "***" if r['p_value'] < 0.001 else ("**" if r['p_value'] < 0.01 else ("*" if r['p_value'] < 0.05 else ""))
                sig_matrix.loc[eco_code, metric] = f"{r['eta_squared']:.3f}{sig_str}"
            else:
                sig_matrix.loc[eco_code, metric] = ""
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(eta_matrix.astype(float), cmap='YlOrRd', annot=sig_matrix.values, fmt='',
                ax=ax, linewidths=0.5, cbar_kws={'label': 'Effect Size (η²)'}, vmin=0, vmax=0.3)
    ax.set_title('Effect Size of Land Cover Differences Within Each Ecoregion\n'
                 '(Kruskal-Wallis η², * p<.05, ** p<.01, *** p<.001)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Metric')
    ax.set_ylabel('Ecoregion')
    ax.set_xticklabels([m.replace('_', '\n') for m in METRICS_TO_TEST], rotation=45, ha='right')
    plt.tight_layout()
    eta_heatmap_path = os.path.join(lcxeco_plots_dir, "statistical_tests_lc_x_ecoregion_effect_sizes.png")
    plt.savefig(eta_heatmap_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ {eta_heatmap_path}")

# CSVs
print("\nSpeichere statistische Testergebnisse...")
kw_rows = []
for metric in METRICS_TO_TEST:
    if metric in kw_results_lc:
        kw = kw_results_lc[metric]
        kw_rows.append({'Metric': metric, 'Metric_Label': METRIC_LABELS[metric],
                        'H_Statistic': kw['H_statistic'], 'p_value': kw['p_value'],
                        'Eta_Squared': kw['eta_squared'], 'Effect_Size': kw['effect_size'],
                        'k_Groups': kw['k_groups'], 'n_Total': kw['n_total'],
                        'Significant_0.05': kw['significant']})
pd.DataFrame(kw_rows).to_csv(os.path.join(lc_csv_dir, "statistical_tests_landcover_kruskal_wallis.csv"), index=False)
print(f"  ✓ statistical_tests_landcover_kruskal_wallis.csv")

eco_kw_rows = []
for eco_code in sorted(kw_results_lc_per_eco.keys()):
    for metric in METRICS_TO_TEST:
        if metric in kw_results_lc_per_eco[eco_code]:
            r = kw_results_lc_per_eco[eco_code][metric]
            eco_kw_rows.append({'Ecoregion': eco_code, 'Metric': metric,
                                'H_Statistic': r['H_statistic'], 'p_value': r['p_value'],
                                'Eta_Squared': r['eta_squared'], 'Effect_Size': r['effect_size'],
                                'k_Groups': r['k_groups'], 'n_Total': r['n_total'],
                                'Significant_0.05': r['significant']})
pd.DataFrame(eco_kw_rows).to_csv(os.path.join(lcxeco_csv_dir, "statistical_tests_lc_x_ecoregion_kruskal_wallis.csv"), index=False)
print(f"  ✓ statistical_tests_lc_x_ecoregion_kruskal_wallis.csv")

if HAS_POSTHOC and dunn_results_lc:
    for metric, dunn_df in dunn_results_lc.items():
        dunn_df.to_csv(os.path.join(lc_csv_dir, f"statistical_tests_landcover_dunn_{metric}.csv"))
        print(f"  ✓ statistical_tests_landcover_dunn_{metric}.csv")

median_rows = []
for lc_name in lc_names_ordered:
    if lc_name not in lc_pixel_metrics:
        continue
    row = {'LC_Major': lc_name, 'N_Pixels_Sampled': lc_pixel_metrics[lc_name]['n_pixels_sampled']}
    for metric in METRICS_TO_TEST:
        vals = lc_pixel_metrics[lc_name][metric]
        row[f'{metric}_median'] = np.nanmedian(vals)
        row[f'{metric}_iqr'] = stats.iqr(vals[~np.isnan(vals)]) if np.sum(~np.isnan(vals)) > 0 else np.nan
    median_rows.append(row)
pd.DataFrame(median_rows).to_csv(os.path.join(lc_csv_dir, "statistical_tests_landcover_medians.csv"), index=False)
print(f"  ✓ statistical_tests_landcover_medians.csv")

print(f"\n✓ Alle statistischen Tests für Landcover abgeschlossen!")



3b. STATISTISCHE TESTS – LANDCOVER-VERGLEICHE
----------------------------------------------------------------------
✓ scikit-posthocs verfügbar

TEST A: Kruskal-Wallis – Major Landcover Klassen (Europa-weit)

Extrahiere Pixel-Level Metriken pro LC-Klasse...


LC Klassen:   0%|          | 0/6 [00:00<?, ?it/s]C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:61: RuntimeWarning: divide by zero encountered in divide
  fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:61: RuntimeWarning: invalid value encountered in divide
  fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:63: RuntimeWarning: divide by zero encountered in divide
  (recovery_5yr_val - fire_year_val) / fire_loss * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:63: RuntimeWarning: invalid value encountered in divide
  (recovery_5yr_val - fire_year_val) / fire_loss * 100, np.nan)
LC Klassen:  17%|█▋        | 1/6 [00:08<00:43,  8.65s/it]

  ✓ Forests: 157,654 Pixel (sampled: 50,000)


LC Klassen:  33%|███▎      | 2/6 [00:11<00:20,  5.15s/it]

  ✓ Shrublands: 12,943 Pixel (sampled: 12,943)


...

  ✓ Croplands: 1,466,288 Pixel (sampled: 50,000)

  Pre-Fire Woody Cover (Mean, %):
    H = 139706.1, p = 0.00e+00 ***
    η² = 0.6313 (large)

  Pre-Fire Trend (Slope, %/yr):
    H = 4868.7, p = 0.00e+00 ***
    η² = 0.0220 (small)

  Absolute Fire Loss (%):
    H = 1101.8, p = 5.47e-236 ***
    η² = 0.0050 (negligible)

  Relative Fire Loss (%):
    H = 883.7, p = 8.96e-189 ***
    η² = 0.0042 (negligible)

  5-Year Recovery (% of Loss):
    H = 3619.1, p = 0.00e+00 ***
    η² = 0.0617 (medium)

  Dunn's Post-hoc Tests (Bonferroni):
    pre_fire_mean: 15/15 signifikante Paare
    pre_fire_trend: 13/15 signifikante Paare
    fire_loss: 13/15 signifikante Paare
    fire_loss_pct: 13/15 signifikante Paare
    recovery_5yr_pct: 14/15 signifikante Paare

TEST B: Kruskal-Wallis – LC-Klassen innerhalb jeder Ecoregion


Ecoregions:   0%|          | 0/11 [00:00<?, ?it/s]C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:61: RuntimeWarning: divide by zero encountered in divide
  fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:61: RuntimeWarning: invalid value encountered in divide
  fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:63: RuntimeWarning: divide by zero encountered in divide
  (recovery_5yr_val - fire_year_val) / fire_loss * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:63: RuntimeWarning: invalid value encountered in divide
  (recovery_5yr_val - fire_year_val) / fire_loss * 100, np.nan)
Ecoregions: 100%|██████████| 11/11 [02:12<00:00, 12.08s/it]



Erstelle Visualisierungen...
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Landcover\plots\statistical_tests_landcover_boxplots.png


C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:233: RuntimeWarning: divide by zero encountered in log10
  log_p = np.clip(-np.log10(dunn_df.values.astype(float)), 0, 50)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:233: RuntimeWarning: divide by zero encountered in log10
  log_p = np.clip(-np.log10(dunn_df.values.astype(float)), 0, 50)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:233: RuntimeWarning: divide by zero encountered in log10
  log_p = np.clip(-np.log10(dunn_df.values.astype(float)), 0, 50)


  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Landcover\plots\statistical_tests_landcover_dunn_heatmaps.png
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\LC_x_Ecoregion\plots\statistical_tests_lc_x_ecoregion_effect_sizes.png

Speichere statistische Testergebnisse...
  ✓ statistical_tests_landcover_kruskal_wallis.csv
  ✓ statistical_tests_lc_x_ecoregion_kruskal_wallis.csv
  ✓ statistical_tests_landcover_dunn_pre_fire_mean.csv
  ✓ statistical_tests_landcover_dunn_pre_fire_trend.csv
  ✓ statistical_tests_landcover_dunn_fire_loss.csv
  ✓ statistical_tests_landcover_dunn_fire_loss_pct.csv
  ✓ statistical_tests_landcover_dunn_recovery_5yr_pct.csv
  ✓ statistical_tests_landcover_medians.csv

✓ Alle statistischen Tests für Landcover abgeschlossen!


In [2]:
# === SCHRITT 3c: ECOREGION-VISUALISIERUNGEN ===
# (Ersetzt Schritte 7–15 aus 04_ecoregions_MBA_analysis.ipynb)

print("\n3c. ECOREGION-VISUALISIERUNGEN")
print("-" * 70)

# =====================================================
# PLOT E1: Alle Ecoregion-Trajektorien (1 Fire)
# =====================================================
print("\nPlot E1: Alle Ecoregion-Trajektorien (1 Fire)...")

fig, ax = plt.subplots(figsize=(14, 8))

for eco_code, data in sorted(eco_results.items()):
    color = data['hex_color']
    traj = np.array(data['trajectory'])
    std = np.array(data['std'])
    
    ax.plot(rel_years, traj, marker='o', linewidth=2.5,
            label=f"{eco_code} – {data['name']} (n={data['n_pixels']:,})",
            color=color, alpha=0.9)
    
    lower = np.maximum(traj - std, 0)
    upper = np.minimum(traj + std, 100)
    ax.fill_between(rel_years, lower, upper, alpha=0.1, color=color)

ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7, label='Fire Year')
ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)

ax.annotate('Pre-Fire', xy=(-3, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkgreen', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))
ax.annotate('Fire\nYear', xy=(0.5, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkred', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='mistyrose', alpha=0.5))
ax.annotate('Recovery', xy=(5.5, ax.get_ylim()[1]*0.95), fontsize=10, ha='center',
            color='darkblue', fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

ax.set_xlabel('Years Relative to Fire Event', fontsize=12)
ax.set_ylabel('Woody Cover (%)', fontsize=12)
ax.set_title('Post-Fire Woody Cover Trajectories by Ecoregion\n(Single Fire Events Only)',
             fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(-5.5, 10.5)

plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "eco_trajectories_1fire_all.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_trajectories_1fire_all.png")

# =====================================================
# PLOT E2: Subplots pro Ecoregion (1 Fire)
# =====================================================
print("\nPlot E2: Subplots pro Ecoregion...")

n_eco = len(eco_results)
n_cols = 3
n_rows = int(np.ceil(n_eco / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes = axes.flatten()

for i, (eco_code, data) in enumerate(sorted(eco_results.items())):
    ax = axes[i]
    color = data['hex_color']
    traj = np.array(data['trajectory'])
    std = np.array(data['std'])
    
    lower = np.maximum(traj - std, 0)
    upper = np.minimum(traj + std, 100)
    
    ax.plot(rel_years, traj, marker='o', linewidth=2.5, color=color)
    ax.fill_between(rel_years, lower, upper, alpha=0.3, color=color)
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.7)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.7)
    
    ax.set_ylim(max(0, np.nanmin(lower) - 5), min(100, np.nanmax(upper) + 5))
    ax.set_title(f"{eco_code} – {data['name']}\n(n={data['n_pixels']:,})",
                 fontsize=10, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire', fontsize=9)
    ax.set_ylabel('Woody Cover (%)', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2)

for i in range(n_eco, len(axes)):
    axes[i].axis('off')

plt.suptitle('Woody Cover Trajectories by Ecoregion (Single Fire Events)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "eco_trajectories_1fire_panel.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_trajectories_1fire_panel.png")

# =====================================================
# PLOT E3: Ecoregion × Fire Count
# =====================================================
print("\nPlot E3: Ecoregion × Fire Count...")

eco_with_fc = {e: d for e, d in eco_firecount_results.items() if d}
n_eco_fc = len(eco_with_fc)
n_cols = 3
n_rows = int(np.ceil(n_eco_fc / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
axes = axes.flatten()

plot_idx = 0
for eco_code in sorted(eco_with_fc.keys()):
    fc_data = eco_with_fc[eco_code]
    ax = axes[plot_idx]
    eco_name = eco_results.get(eco_code, {}).get('name', eco_code)
    
    for fc, data in sorted(fc_data.items()):
        traj = np.array(data['trajectory'])
        std = np.array(data['std'])
        color = FIRE_COUNT_COLORS[fc]
        
        ax.plot(rel_years, traj, marker='o', linewidth=2.5,
                color=color,
                label=f'{fc} Fire(s) (n={data["n_pixels"]:,})', alpha=0.8)
        
        lower = np.maximum(traj - std, 0)
        upper = np.minimum(traj + std, 100)
        ax.fill_between(rel_years, lower, upper, alpha=0.15, color=color)
    
    ax.axvline(x=0, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax.axvline(x=1, color='red', linestyle='--', linewidth=2, alpha=0.5)
    ax.set_title(f"{eco_code} – {eco_name}", fontsize=10, fontweight='bold')
    ax.set_xlabel('Years Relative to Fire', fontsize=9)
    ax.set_ylabel('Woody Cover (%)', fontsize=9)
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-5.5, 10.5)
    
    eco_color = eco_results.get(eco_code, {}).get('hex_color', '#808080')
    for spine in ax.spines.values():
        spine.set_edgecolor(eco_color)
        spine.set_linewidth(2)
    
    plot_idx += 1

for i in range(plot_idx, len(axes)):
    axes[i].axis('off')

plt.suptitle('Woody Cover Trajectories by Ecoregion × Fire Count',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "eco_trajectories_by_fire_count.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_trajectories_by_fire_count.png")

# =====================================================
# PLOT E4: Jährlich verbrannte Fläche (gesamt)
# =====================================================
print("\nPlot E4: Jährlich verbrannte Fläche...")

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(burned_area_total['Year'], burned_area_total['Burned_km2'],
       color='#d62728', alpha=0.8, edgecolor='darkred', linewidth=0.5)

mean_val = burned_area_total['Burned_km2'].mean()
ax.axhline(y=mean_val, color='black', linestyle='--', linewidth=1.5,
           label=f'Mean: {mean_val:,.0f} km²/yr')

# Werte auf Bars
for _, row in burned_area_total.iterrows():
    ax.text(row['Year'], row['Burned_km2'], f'{row["Burned_km2"]:.0f}',
            ha='center', va='bottom', fontsize=7, rotation=90)

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Burned Area (km²)', fontsize=12)
ax.set_title('Annual Burned Area – Entire Study Area', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "burned_area_annual_total.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ burned_area_annual_total.png")

# =====================================================
# PLOT E5: Verbrannte Fläche pro Ecoregion (absolut + prozentual)
# =====================================================
print("\nPlot E5: Burned Area pro Ecoregion (absolut + %)...")

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Pivot für Line-Plots
pivot_abs = burned_area_eco.pivot_table(
    index='Year', columns='Ecoregion', values='Burned_km2', fill_value=0
)
pivot_pct = burned_area_eco.pivot_table(
    index='Year', columns='Ecoregion', values='Burned_Fraction', fill_value=0
) * 100  # → Prozent

for eco_code in sorted(pivot_abs.columns):
    color = eco_results.get(eco_code, {}).get('hex_color', '#808080')
    axes[0].plot(pivot_abs.index, pivot_abs[eco_code], marker='o', linewidth=2,
                 label=eco_code, color=color, alpha=0.85)
    axes[1].plot(pivot_pct.index, pivot_pct[eco_code], marker='o', linewidth=2,
                 label=eco_code, color=color, alpha=0.85)

axes[0].set_title('Annual Burned Area per Ecoregion (absolute)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Burned Area (km²)', fontsize=12)
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Annual Burned Area per Ecoregion (% of Region)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Burned Area (%)', fontsize=12)
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
axes[1].grid(True, alpha=0.3)

for ax in axes:
    ax.set_xlabel('Year', fontsize=12)

plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "burned_area_by_ecoregion.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ burned_area_by_ecoregion.png")

# =====================================================
# PLOT E6: Fire Season Length – Range + Boxplot + Heatmap + Trend
# =====================================================
print("\nPlot E6: Fire Season Length (4 Subplots)...")

fig, axes = plt.subplots(1, 4, figsize=(30, 7),
                         gridspec_kw={'width_ratios': [1, 1.2, 2.5, 1]})

# Gemeinsame Reihenfolge: alphabetisch, ohne "Outside"
ECO_PLOT_ORDER = sorted([
    eco for eco in fire_stats_summary['Ecoregion'].values
    if eco != 'Outside'
])
ECO_REV        = ECO_PLOT_ORDER[::-1]   # A oben, Z unten (für Range & Boxplot)
y_pos          = np.arange(len(ECO_PLOT_ORDER))
n_ecos         = len(ECO_PLOT_ORDER)
eco_code_to_id = {v['code']: k for k, v in eco_mapping.items()}

# E6a) Range – invertiert (A oben)
ax = axes[0]
for i, eco_code in enumerate(ECO_REV):
    row   = fire_stats_summary[fire_stats_summary['Ecoregion'] == eco_code].iloc[0]
    color = eco_results.get(eco_code, {}).get('hex_color', '#808080')
    ax.plot([row['Season_Length_Min'], row['Season_Length_Max']], [i, i],
            'o-', linewidth=2, markersize=8, color=color)
    ax.plot(row['Season_Length_Mean'], i, 'D', markersize=10, color='darkred', zorder=3)
ax.set_yticks(y_pos)
ax.set_yticklabels(ECO_REV)
ax.set_xlabel('Fire Season Length (Months)', fontsize=11)
ax.set_title('Fire Season Length Range\n(Min-Mean-Max)', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# E6b) Boxplot – invertiert (A oben)
ax = axes[1]
season_box_data   = []
season_box_labels = []
season_box_colors = []
for eco_code in ECO_REV:
    eco_id = eco_code_to_id.get(eco_code)
    if eco_id is None:
        continue
    eco_mask   = (eco_raster == eco_id)
    all_season = mba_season[:, eco_mask].flatten()
    all_season = all_season[(all_season > 0) & (~np.isnan(all_season))]
    if len(all_season) > 0:
        season_box_data.append(all_season)
        season_box_labels.append(eco_code)
        season_box_colors.append(eco_results.get(eco_code, {}).get('hex_color', '#808080'))
bp = ax.boxplot(season_box_data, labels=season_box_labels, patch_artist=True,
                showfliers=False, vert=False)
for j, color in enumerate(season_box_colors):
    bp['boxes'][j].set_facecolor(color)
    bp['boxes'][j].set_alpha(0.6)
ax.set_xlim(0.5, 12.5)
ax.set_xticks(range(1, 13))
ax.set_xlabel('Fire Season Length (Months)', fontsize=11)
ax.set_title('Fire Season Length\nDistribution', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# E6c) Heatmap Frequency – Zeilen: ECO_PLOT_ORDER (A=Zeile 0=oben), Spalten: Monat 1–12
ax = axes[2]
heatmap_season = np.full((n_ecos, 12), np.nan)
for i, eco_code in enumerate(ECO_PLOT_ORDER):
    eco_id = eco_code_to_id.get(eco_code)
    if eco_id is None:
        continue
    eco_mask   = (eco_raster == eco_id)
    all_season = mba_season[:, eco_mask].flatten()
    all_season = all_season[(all_season > 0) & (~np.isnan(all_season))]
    if len(all_season) == 0:
        continue
    unique_vals, cnts = np.unique(all_season.astype(int), return_counts=True)
    total = len(all_season)
    for u, c in zip(unique_vals, cnts):
        if 1 <= u <= 12:
            heatmap_season[i, u - 1] = c / total * 100

cmap_s = plt.cm.YlOrRd.copy()
cmap_s.set_bad(color='white')
masked_s = np.ma.masked_invalid(heatmap_season)
im = ax.imshow(masked_s, cmap=cmap_s, aspect='auto', vmin=0, vmax=100, origin='upper')
for row_i in range(n_ecos):
    for col_j in range(12):
        val = heatmap_season[row_i, col_j]
        if not np.isnan(val) and val > 0:
            fmt        = f"{val:.3f}" if val < 0.1 else f"{val:.1f}"
            text_color = 'white' if val > 60 else 'black'
            ax.text(col_j, row_i, f"{fmt}%", ha='center', va='center',
                    fontsize=6, color=text_color)
ax.set_xticks(range(12))
ax.set_xticklabels(range(1, 13))
ax.set_yticks(range(n_ecos))
ax.set_yticklabels(ECO_PLOT_ORDER)
ax.set_xlabel('Fire Season Length (Months)', fontsize=11)
ax.set_title('Season Length Frequency\n(% of pixel-years per ecoregion)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='%', shrink=0.8)

# E6d) Temporal Trend
ax = axes[3]
yearly_season_means = []
yearly_season_max   = []
for yr_idx in range(len(years_burned)):
    vals  = mba_season[yr_idx]
    valid = vals[(vals > 0) & (~np.isnan(vals))]
    yearly_season_means.append(float(np.mean(valid)) if len(valid) > 0 else np.nan)
    yearly_season_max.append(float(np.max(valid))    if len(valid) > 0 else np.nan)
ax.plot(years_burned, yearly_season_means, marker='o', linewidth=2, color='orangered', label='Mean')
ax.plot(years_burned, yearly_season_max,   marker='s', linewidth=2, color='darkred',
        label='Maximum', linestyle='--')
ax.fill_between(years_burned, yearly_season_means, alpha=0.3, color='orangered')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Season Length (Months)', fontsize=11)
ax.set_title('Temporal Trend:\nSeason Length', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Fire Season Length Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "fire_season_length_analysis.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ fire_season_length_analysis.png")

# =====================================================
# PLOT E7: Fire Count – Range + Boxplot + Heatmap + Trend
# =====================================================
print("\nPlot E7: Fire Count (4 Subplots)...")

fig, axes = plt.subplots(1, 4, figsize=(26, 7),
                         gridspec_kw={'width_ratios': [1, 1.2, 1.5, 1]})
# ECO_PLOT_ORDER, ECO_REV, y_pos, n_ecos, eco_code_to_id — von E6 übernommen

# E7a) Range – invertiert (A oben)
ax = axes[0]
for i, eco_code in enumerate(ECO_REV):
    row   = fire_stats_summary[fire_stats_summary['Ecoregion'] == eco_code].iloc[0]
    color = eco_results.get(eco_code, {}).get('hex_color', '#808080')
    ax.plot([row['Fire_Count_Min'], row['Fire_Count_Max']], [i, i],
            'o-', linewidth=2, markersize=8, color=color)
    ax.plot(row['Fire_Count_Mean'], i, 'D', markersize=10, color='darkorange', zorder=3)
ax.set_yticks(y_pos)
ax.set_yticklabels(ECO_REV)
ax.set_xlabel('Number of Fire Events', fontsize=11)
ax.set_title('Fire Count Range\n(Min-Mean-Max)', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# E7b) Boxplot – invertiert (A oben)
ax = axes[1]
count_box_data   = []
count_box_labels = []
count_box_colors = []
for eco_code in ECO_REV:
    eco_id = eco_code_to_id.get(eco_code)
    if eco_id is None:
        continue
    eco_mask  = (eco_raster == eco_id)
    all_count = mba_count[:, eco_mask].flatten()
    all_count = all_count[(all_count > 0) & (~np.isnan(all_count))]
    if len(all_count) > 0:
        count_box_data.append(all_count)
        count_box_labels.append(eco_code)
        count_box_colors.append(eco_results.get(eco_code, {}).get('hex_color', '#808080'))
bp = ax.boxplot(count_box_data, labels=count_box_labels, patch_artist=True,
                showfliers=False, vert=False)
for j, color in enumerate(count_box_colors):
    bp['boxes'][j].set_facecolor(color)
    bp['boxes'][j].set_alpha(0.6)
ax.set_xlim(0.5, 4.5)
ax.set_xticks(range(1, 5))
ax.set_xlabel('Number of Fire Events', fontsize=11)
ax.set_title('Fire Count\nDistribution', fontsize=12, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# E7c) Heatmap Frequency – Zeilen: ECO_PLOT_ORDER (A=Zeile 0=oben), Spalten: 1–4
ax = axes[2]
heatmap_count = np.full((n_ecos, 4), np.nan)
for i, eco_code in enumerate(ECO_PLOT_ORDER):
    eco_id = eco_code_to_id.get(eco_code)
    if eco_id is None:
        continue
    eco_mask  = (eco_raster == eco_id)
    all_count = mba_count[:, eco_mask].flatten()
    all_count = all_count[(all_count > 0) & (~np.isnan(all_count))]
    if len(all_count) == 0:
        continue
    unique_vals, cnts = np.unique(all_count.astype(int), return_counts=True)
    total = len(all_count)
    for u, c in zip(unique_vals, cnts):
        if 1 <= u <= 4:
            heatmap_count[i, u - 1] = c / total * 100

cmap_c = plt.cm.YlOrRd.copy()
cmap_c.set_bad(color='white')
masked_c = np.ma.masked_invalid(heatmap_count)
im2 = ax.imshow(masked_c, cmap=cmap_c, aspect='auto', vmin=0, vmax=100, origin='upper')
for row_i in range(n_ecos):
    for col_j in range(4):
        val = heatmap_count[row_i, col_j]
        if not np.isnan(val) and val > 0:
            fmt        = f"{val:.3f}" if val < 0.1 else f"{val:.1f}"
            text_color = 'white' if val > 60 else 'black'
            ax.text(col_j, row_i, f"{fmt}%", ha='center', va='center',
                    fontsize=8, color=text_color)
ax.set_xticks(range(4))
ax.set_xticklabels(range(1, 5))
ax.set_yticks(range(n_ecos))
ax.set_yticklabels(ECO_PLOT_ORDER)
ax.set_xlabel('Number of Fire Events', fontsize=11)
ax.set_title('Fire Count Frequency\n(% of pixel-years per ecoregion)', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=ax, label='%', shrink=0.8)

# E7d) Temporal Trend
ax = axes[3]
yearly_count_means = []
yearly_count_max   = []
for yr_idx in range(len(years_burned)):
    vals  = mba_count[yr_idx]
    valid = vals[(vals > 0) & (~np.isnan(vals))]
    yearly_count_means.append(float(np.mean(valid)) if len(valid) > 0 else np.nan)
    yearly_count_max.append(float(np.max(valid))    if len(valid) > 0 else np.nan)
ax.plot(years_burned, yearly_count_means, marker='o', linewidth=2, color='darkorange', label='Mean')
ax.plot(years_burned, yearly_count_max,   marker='s', linewidth=2, color='chocolate',
        label='Maximum', linestyle='--')
ax.fill_between(years_burned, yearly_count_means, alpha=0.3, color='orange')
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Fire Count', fontsize=11)
ax.set_title('Temporal Trend:\nFire Count', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Fire Count Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "fire_count_analysis.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ fire_count_analysis.png")

# # =====================================================
# # PLOT E6: Fire Season Length – Range + Distribution + Trend
# # =====================================================
# print("\nPlot E6: Fire Season Length (3 Subplots)...")

# fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# # Gemeinsame Ecoregion-Reihenfolge für E6 & E7: alphabetisch, ohne "Outside"
# ECO_PLOT_ORDER = sorted([
#     eco for eco in fire_stats_summary['Ecoregion'].values
#     if eco != 'Outside'
# ])
# y_pos = np.arange(len(ECO_PLOT_ORDER))

# # Reverse-Mapping eco_code → eco_id (für Boxplot-Extraktion)
# eco_code_to_id = {v['code']: k for k, v in eco_mapping.items()}

# # E6a) Range Plot (Min-Mean-Max)
# ax = axes[0]
# for i, eco_code in enumerate(ECO_PLOT_ORDER):
#     row = fire_stats_summary[fire_stats_summary['Ecoregion'] == eco_code].iloc[0]
#     color = eco_results.get(eco_code, {}).get('hex_color', '#808080')
#     ax.plot([row['Season_Length_Min'], row['Season_Length_Max']], [i, i],
#             'o-', linewidth=2, markersize=8, color=color)
#     ax.plot(row['Season_Length_Mean'], i, 'D', markersize=10, color='darkred', zorder=3)

# ax.set_yticks(y_pos)
# ax.set_yticklabels(ECO_PLOT_ORDER)
# ax.set_xlabel('Fire Season Length (Months)', fontsize=11)
# ax.set_title('Fire Season Length Range\n(Min-Mean-Max)', fontsize=12, fontweight='bold')
# ax.grid(axis='x', alpha=0.3)

# # E6b) Box Plot Distribution – gleiche Reihenfolge, x-Achse 1–12 ganzzahlig
# ax = axes[1]
# season_box_data   = []
# season_box_labels = []
# season_box_colors = []

# for eco_code in ECO_PLOT_ORDER:
#     eco_id = eco_code_to_id.get(eco_code)
#     if eco_id is None:
#         continue
#     eco_mask   = (eco_raster == eco_id)
#     all_season = mba_season[:, eco_mask].flatten()
#     all_season = all_season[(all_season > 0) & (~np.isnan(all_season))]
#     if len(all_season) > 0:
#         season_box_data.append(all_season)
#         season_box_labels.append(eco_code)
#         season_box_colors.append(eco_results.get(eco_code, {}).get('hex_color', '#808080'))

# bp = ax.boxplot(season_box_data, labels=season_box_labels, patch_artist=True,
#                 showfliers=False, vert=False)
# for j, color in enumerate(season_box_colors):
#     bp['boxes'][j].set_facecolor(color)
#     bp['boxes'][j].set_alpha(0.6)

# ax.set_xlim(0.5, 12.5)
# ax.set_xticks(range(1, 13))
# ax.set_xlabel('Fire Season Length (Months)', fontsize=11)
# ax.set_title('Fire Season Length Distribution', fontsize=12, fontweight='bold')
# ax.grid(axis='x', alpha=0.3)

# # E6c) Temporal Trend  ← unverändert
# ax = axes[2]
# yearly_season_means = []
# yearly_season_max   = []
# for yr_idx in range(len(years_burned)):
#     vals  = mba_season[yr_idx]
#     valid = vals[(vals > 0) & (~np.isnan(vals))]
#     yearly_season_means.append(float(np.mean(valid)) if len(valid) > 0 else np.nan)
#     yearly_season_max.append(float(np.max(valid))    if len(valid) > 0 else np.nan)

# ax.plot(years_burned, yearly_season_means, marker='o', linewidth=2, color='orangered', label='Mean')
# ax.plot(years_burned, yearly_season_max,   marker='s', linewidth=2, color='darkred',
#         label='Maximum', linestyle='--')
# ax.fill_between(years_burned, yearly_season_means, alpha=0.3, color='orangered')
# ax.set_xlabel('Year', fontsize=11)
# ax.set_ylabel('Season Length (Months)', fontsize=11)
# ax.set_title('Temporal Trend: Season Length', fontsize=12, fontweight='bold')
# ax.legend(fontsize=10)
# ax.grid(True, alpha=0.3)

# plt.suptitle('Fire Season Length Analysis', fontsize=15, fontweight='bold')
# plt.tight_layout()
# plt.savefig(os.path.join(eco_plots_dir, "fire_season_length_analysis.png"), dpi=300, bbox_inches='tight')
# plt.close()
# print(f"  ✓ fire_season_length_analysis.png")

# # =====================================================
# # PLOT E7: Fire Count – Range + Distribution + Trend
# # =====================================================
# print("\nPlot E7: Fire Count (3 Subplots)...")

# fig, axes = plt.subplots(1, 3, figsize=(22, 7))

# # E7a) Range – gleiche ECO_PLOT_ORDER & y_pos wie E6
# ax = axes[0]
# for i, eco_code in enumerate(ECO_PLOT_ORDER):
#     row = fire_stats_summary[fire_stats_summary['Ecoregion'] == eco_code].iloc[0]
#     color = eco_results.get(eco_code, {}).get('hex_color', '#808080')
#     ax.plot([row['Fire_Count_Min'], row['Fire_Count_Max']], [i, i],
#             'o-', linewidth=2, markersize=8, color=color)
#     ax.plot(row['Fire_Count_Mean'], i, 'D', markersize=10, color='darkorange', zorder=3)

# ax.set_yticks(y_pos)
# ax.set_yticklabels(ECO_PLOT_ORDER)
# ax.set_xlabel('Number of Fire Events', fontsize=11)
# ax.set_title('Fire Count Range\n(Min-Mean-Max)', fontsize=12, fontweight='bold')
# ax.grid(axis='x', alpha=0.3)

# # E7b) Box Plot – gleiche Reihenfolge, x-Achse 1–4 ganzzahlig
# ax = axes[1]
# count_box_data   = []
# count_box_labels = []
# count_box_colors = []

# for eco_code in ECO_PLOT_ORDER:
#     eco_id = eco_code_to_id.get(eco_code)
#     if eco_id is None:
#         continue
#     eco_mask  = (eco_raster == eco_id)
#     all_count = mba_count[:, eco_mask].flatten()
#     all_count = all_count[(all_count > 0) & (~np.isnan(all_count))]
#     if len(all_count) > 0:
#         count_box_data.append(all_count)
#         count_box_labels.append(eco_code)
#         count_box_colors.append(eco_results.get(eco_code, {}).get('hex_color', '#808080'))

# bp = ax.boxplot(count_box_data, labels=count_box_labels, patch_artist=True,
#                 showfliers=False, vert=False)
# for j, color in enumerate(count_box_colors):
#     bp['boxes'][j].set_facecolor(color)
#     bp['boxes'][j].set_alpha(0.6)

# ax.set_xlim(0.5, 4.5)
# ax.set_xticks(range(1, 5))
# ax.set_xlabel('Number of Fire Events', fontsize=11)
# ax.set_title('Fire Count Distribution', fontsize=12, fontweight='bold')
# ax.grid(axis='x', alpha=0.3)

# # E7c) Temporal Trend
# ax = axes[2]
# yearly_count_means = []
# yearly_count_max   = []
# for yr_idx in range(len(years_burned)):
#     vals  = mba_count[yr_idx]
#     valid = vals[(vals > 0) & (~np.isnan(vals))]
#     yearly_count_means.append(float(np.mean(valid)) if len(valid) > 0 else np.nan)
#     yearly_count_max.append(float(np.max(valid))    if len(valid) > 0 else np.nan)

# ax.plot(years_burned, yearly_count_means, marker='o', linewidth=2, color='darkorange', label='Mean')
# ax.plot(years_burned, yearly_count_max,   marker='s', linewidth=2, color='chocolate',
#         label='Maximum', linestyle='--')
# ax.fill_between(years_burned, yearly_count_means, alpha=0.3, color='orange')
# ax.set_xlabel('Year', fontsize=11)
# ax.set_ylabel('Fire Count', fontsize=11)
# ax.set_title('Temporal Trend: Fire Count', fontsize=12, fontweight='bold')
# ax.legend(fontsize=10)
# ax.grid(True, alpha=0.3)

# plt.suptitle('Fire Count Analysis', fontsize=15, fontweight='bold')
# plt.tight_layout()
# plt.savefig(os.path.join(eco_plots_dir, "fire_count_analysis.png"), dpi=300, bbox_inches='tight')
# plt.close()
# print(f"  ✓ fire_count_analysis.png")

# =====================================================
# PLOT E8: Season Length × Fire Count
# =====================================================
print("\nPlot E8: Season Length abhängig von Fire Count...")

fig, ax = plt.subplots(figsize=(14, 8))

# Check if raw values are available
has_raw_data = all(
    len(season_by_fc[c]['values']) > 0
    for c in season_by_fc.keys()
)

if has_raw_data:
    # Full run: proper boxplot
    bp_data   = [season_by_fc[c]['values'] for c in sorted(season_by_fc.keys())]
    bp_labels = [f"{c} Fires\n(n={season_by_fc[c]['n']:,})" for c in sorted(season_by_fc.keys())]

    bp = ax.boxplot(bp_data, labels=bp_labels, patch_artist=True)
    colors_gradient = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(bp_data)))
    for patch, color in zip(bp['boxes'], colors_gradient):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
else:
    # CSV-reconstructed: bar chart with mean values only
    print("  ⚠️  Keine Rohdaten verfügbar – zeige Mittelwerte als Balkendiagramm")
    fc_vals    = sorted(season_by_fc.keys())
    means      = [season_by_fc[c]['mean'] for c in fc_vals]
    bp_labels  = [f"{c} Fires\n(n={season_by_fc[c]['n']:,})" for c in fc_vals]
    x_pos      = np.arange(len(fc_vals))

    colors_gradient = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(fc_vals)))
    bars = ax.bar(x_pos + 1, means, color=colors_gradient, alpha=0.7,
                  edgecolor='black', linewidth=0.8)
    ax.set_xticks(x_pos + 1)
    ax.set_xticklabels(bp_labels)

# Annotate mean values (works for both cases)
for i, fc_val in enumerate(sorted(season_by_fc.keys())):
    y_pos_ann = season_by_fc[fc_val]['mean']
    ax.text(i + 1, y_pos_ann + 0.05, f'{y_pos_ann:.1f}',
            ha='center', va='bottom', fontweight='bold', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

ax.set_xlabel('Number of Fire Events', fontsize=12)
ax.set_ylabel('Fire Season Length (Months)', fontsize=12)
ax.set_title('Fire Season Length by Number of Fire Events', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "fire_season_length_by_count.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ fire_season_length_by_count.png")


# =====================================================
# PLOT E9: Recovery Rates (5 & 10 Jahre) – Heatmap
# =====================================================
print("\nPlot E9: Recovery Rates Heatmap...")

# Berechne Recovery-Metriken pro Ecoregion
eco_recovery_rows = []

for eco_code, data in eco_results.items():
    traj = np.array(data['trajectory'])
    pre_fire = np.nanmean(traj[:5])
    fire_year_val = traj[5]
    loss = pre_fire - fire_year_val
    
    for years_after in [1, 3, 5, 10]:
        idx = 5 + years_after
        if idx < len(traj):
            rec_val = traj[idx]
            rec_pct = ((rec_val - fire_year_val) / loss * 100) if loss > 0 else np.nan
        else:
            rec_val = np.nan
            rec_pct = np.nan
        
        eco_recovery_rows.append({
            'Ecoregion': eco_code,
            'Years_After': years_after,
            'Pre_Fire_Cover': pre_fire,
            'Fire_Year_Cover': fire_year_val,
            'Woody_Loss': loss,
            'Recovery_Cover': rec_val,
            'Recovery_Percent': rec_pct,
            'N_Pixels': data['n_pixels']
        })

df_eco_recovery = pd.DataFrame(eco_recovery_rows)

# Heatmap: Ecoregion × Years_After → Recovery %
pivot_rec = df_eco_recovery.pivot_table(
    index='Ecoregion', columns='Years_After', values='Recovery_Percent'
)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot_rec, cmap='RdYlGn', annot=True, fmt='.0f', ax=ax,
            center=100, linewidths=0.5,
            cbar_kws={'label': 'Recovery (% of Loss)'})
ax.set_title('Woody Cover Recovery by Ecoregion\n(% of Immediate Loss Recovered)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Years After Fire')
ax.set_ylabel('Ecoregion')

plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "eco_recovery_heatmap.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_recovery_heatmap.png")

# =====================================================
# PLOT E10: Summary Heatmap (Eco × Fire Count × Metriken)
# =====================================================
print("\nPlot E10: Comprehensive Summary Heatmap...")

# Berechne Summary-Metriken pro Eco × Fire Count
summary_rows = []

for eco_code, fc_data in eco_firecount_results.items():
    eco_name = eco_results.get(eco_code, {}).get('name', eco_code)
    eco_total_pixels = int(np.sum(eco_raster == eco_results.get(eco_code, {}).get('eco_id', 0)))
    
    for fc, data in fc_data.items():
        traj = np.array(data['trajectory'])
        pre_fire = np.nanmean(traj[:5])
        fire_year_val = traj[5]
        loss = pre_fire - fire_year_val
        
        # 5yr und 10yr Recovery
        rec_5yr = traj[10] if len(traj) > 10 else np.nan
        rec_5yr_pct = ((rec_5yr - fire_year_val) / loss * 100) if loss > 0 else np.nan
        
        rec_10yr = traj[15] if len(traj) > 15 else np.nan
        rec_10yr_pct = ((rec_10yr - fire_year_val) / loss * 100) if loss > 0 else np.nan
        
        summary_rows.append({
            'Ecoregion': eco_code,
            'Ecoregion_Name': eco_name,
            'Fire_Count': fc,
            'N_Pixels': data['n_pixels'],
            'Area_km2': data['n_pixels'] * pixel_area_km2,
            'Pre_Fire_Cover': pre_fire,
            'Fire_Year_Cover': fire_year_val,
            'Woody_Loss': loss,
            'Recovery_5yr_Pct': rec_5yr_pct,
            'Recovery_10yr_Pct': rec_10yr_pct
        })

df_summary = pd.DataFrame(summary_rows)

# 4-Panel Heatmap
metrics_hm = ['Area_km2', 'Woody_Loss', 'Recovery_5yr_Pct', 'Recovery_10yr_Pct']
titles_hm = ['Affected Area (km²)', 'Immediate Loss (%)', 
             '5-Year Recovery (%)', '10-Year Recovery (%)']
cmaps_hm = ['YlOrRd', 'Reds', 'RdYlGn', 'RdYlGn']
centers_hm = [None, None, 100, 100]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
axes = axes.flatten()

for idx, (metric, title, cmap, center) in enumerate(zip(metrics_hm, titles_hm, cmaps_hm, centers_hm)):
    ax = axes[idx]
    
    pivot = df_summary.pivot_table(
        index='Fire_Count', columns='Ecoregion', values=metric
    )
    
    sns.heatmap(pivot, cmap=cmap, annot=True, 
                fmt='.1f' if 'Area' in metric else '.0f',
                ax=ax, linewidths=0.5, center=center,
                cbar_kws={'label': title})
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Fire Count')
    ax.set_yticklabels([f'{int(fc)} Fire(s)' for fc in pivot.index])

plt.suptitle('Comprehensive Summary: Ecoregion × Fire Count',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "eco_summary_heatmap.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ eco_summary_heatmap.png")

print(f"\n✓ Alle Ecoregion-Visualisierungen erstellt!")
print(f"  Plots gespeichert in: {eco_plots_dir}")



3c. ECOREGION-VISUALISIERUNGEN
----------------------------------------------------------------------

Plot E1: Alle Ecoregion-Trajektorien (1 Fire)...


NameError: name 'plt' is not defined

Schritt 3c: Ecoregion-Visualisierungen (Plots E1–E10)
E1 – eco_trajectories_1fire_all.png

Figure 9. Mean woody cover trajectories (±1 SD) relative to a single fire event, stratified by ecoregion. Line colours correspond to the ecoregion colour scheme. Year 0 denotes the fire year; the pre-fire window spans years −5 to −1 and the recovery period extends to +10 years.

E2 – eco_trajectories_1fire_panel.png

Figure 10. Individual woody cover trajectories (mean ±1 SD) for each ecoregion following a single fire event. Coloured panel borders correspond to the ecoregion colour scheme. Y-axis ranges are adjusted per panel to highlight regional dynamics.

E3 – eco_trajectories_by_fire_count.png

Figure 11. Woody cover trajectories stratified by ecoregion and cumulative fire count (1–4 fires). Each panel represents one ecoregion; line colours indicate the number of fire events. The first fire event serves as the temporal reference point (year 0).

E4 – burned_area_annual_total.png

Figure 12. Annual burned area (km²) across the entire study area from 2000 to 2025. The dashed horizontal line indicates the long-term mean. Individual bar labels show the annual burned area value.

E5 – burned_area_by_ecoregion.png

Figure 13. Annual burned area by ecoregion shown as (a) absolute values in km² and (b) as a fraction of the total ecoregion area (%). Line colours correspond to the ecoregion colour scheme.

E6 – fire_season_length_analysis.png

Figure 14. Fire season length analysis across ecoregions. (a) Range plot showing minimum, mean (diamond), and maximum season length per ecoregion. (b) Box plot distribution of fire season length across all burned pixels per ecoregion. (c) Temporal trend of mean and maximum fire season length across the entire study area (2000–2025).

E7 – fire_count_analysis.png

Figure 15. Fire count analysis across ecoregions. (a) Range plot showing minimum, mean (diamond), and maximum number of fire events per ecoregion. (b) Box plot distribution of fire counts across all burned pixels per ecoregion. (c) Temporal trend of mean and maximum fire counts across the study area (2000–2025).

E8 – fire_season_length_by_count.png

Figure 16. Fire season length as a function of cumulative fire count. Box plots show the distribution of season length (months) for pixels that experienced 1, 2, 3, or more fire events. Mean values are annotated above each box. Colours follow a sequential gradient from yellow to red.

E9 – eco_recovery_heatmap.png

Figure 17. Heatmap of woody cover recovery (% of immediate loss recovered) by ecoregion at 1, 3, 5, and 10 years after a single fire event. Values of 100% indicate full recovery to pre-fire loss levels; values exceeding 100% indicate overcompensation. Colour scale is centred at 100%.

E10 – eco_summary_heatmap.png

Figure 18. Comprehensive summary of fire impact across ecoregions and fire counts. Four panels show (a) affected area in km², (b) immediate woody cover loss (percentage points), (c) 5-year recovery (% of loss), and (d) 10-year recovery (% of loss). Rows represent fire count categories; columns represent ecoregions.



In [32]:
# === SCHRITT 3d: STATISTISCHE TESTS – ECOREGION-VERGLEICHE ===

print("\n3d. STATISTISCHE TESTS – ECOREGION-VERGLEICHE")
print("-" * 70)

# Extrahiere Pixel-Level Metriken pro Ecoregion
print("\nExtrahiere Pixel-Level Metriken pro Ecoregion...")

eco_pixel_metrics = {}

for eco_id in tqdm(unique_eco_ids, desc="Ecoregions"):
    eco_code = eco_mapping[eco_id]['code']
    eco_mask = (eco_raster == eco_id)
    
    metrics = extract_pixel_metrics_lc(
        woody, burned, fire_counts, first_fire_idx, eco_mask,
        years_woody, years_burned, nodata,
        max_samples=MAX_SAMPLES
    )
    
    if metrics is not None:
        eco_pixel_metrics[eco_code] = metrics
        print(f"  ✓ {eco_code}: {metrics['n_pixels_total']:,} Pixel "
              f"(sampled: {metrics['n_pixels_sampled']:,})")

# Kruskal-Wallis Tests zwischen Ecoregions
eco_names_ordered = sorted(eco_pixel_metrics.keys())

kw_results_eco = {}

for metric in METRICS_TO_TEST:
    groups = [eco_pixel_metrics[eco][metric] for eco in eco_names_ordered]
    result = kruskal_wallis_with_effect_size(groups, eco_names_ordered, metric)
    
    if result is not None:
        kw_results_eco[metric] = result
        sig_str = "***" if result['p_value'] < 0.001 else (
            "**" if result['p_value'] < 0.01 else (
            "*" if result['p_value'] < 0.05 else "n.s."))
        print(f"\n  {METRIC_LABELS[metric]}:")
        print(f"    H = {result['H_statistic']:.1f}, p = {result['p_value']:.2e} {sig_str}")
        print(f"    η² = {result['eta_squared']:.4f} ({result['effect_size']})")

# Dunn's Post-hoc
dunn_results_eco = {}

if HAS_POSTHOC:
    print("\n  Dunn's Post-hoc Tests (Bonferroni) – Ecoregions:")
    for metric in METRICS_TO_TEST:
        if metric not in kw_results_eco or not kw_results_eco[metric]['significant']:
            continue
        
        result = kw_results_eco[metric]
        all_values = []
        all_labels = []
        for g, name in zip(result['groups'], result['group_names']):
            all_values.extend(g.tolist())
            all_labels.extend([name] * len(g))
        
        # Erstelle DataFrame statt numpy-Arrays zu übergeben
        dunn_input_df = pd.DataFrame({
            'value': all_values,
            'group': all_labels
        })
        
        dunn_df = sp.posthoc_dunn(
            a=dunn_input_df,
            val_col='value',
            group_col='group',
            p_adjust='bonferroni'
        )
        dunn_results_eco[metric] = dunn_df
        
        n_pairs = dunn_df.shape[0] * (dunn_df.shape[0] - 1) // 2
        sig_pairs = np.sum(dunn_df.values[np.triu_indices_from(dunn_df.values, k=1)] < 0.05)
        print(f"    {metric}: {sig_pairs}/{n_pairs} signifikante Paare")

# --- Boxplots: Metriken nach Ecoregion ---
print("\nErstelle Ecoregion-Boxplots...")

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes_flat = axes.flatten()

for i, metric in enumerate(METRICS_TO_TEST):
    ax = axes_flat[i]
    
    box_data = []
    box_labels = []
    box_colors = []
    
    for eco_code in eco_names_ordered:
        vals = eco_pixel_metrics[eco_code][metric]
        valid = vals[~np.isnan(vals)]
        if len(valid) > 0:
            box_data.append(valid)
            box_labels.append(eco_code)
            box_colors.append(eco_results.get(eco_code, {}).get('hex_color', '#808080'))
    
    bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True,
                    showfliers=False, widths=0.6)
    
    for j, color in enumerate(box_colors):
        bp['boxes'][j].set_facecolor(color)
        bp['boxes'][j].set_alpha(0.6)
    
    ax.set_title(METRIC_LABELS[metric], fontsize=11, fontweight='bold')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    
    if metric in kw_results_eco:
        kw = kw_results_eco[metric]
        sig_str = "***" if kw['p_value'] < 0.001 else (
            "**" if kw['p_value'] < 0.01 else (
            "*" if kw['p_value'] < 0.05 else "n.s."))
        ax.text(0.02, 0.98, f"KW: H={kw['H_statistic']:.0f}, η²={kw['eta_squared']:.3f} {sig_str}",
                transform=ax.transAxes, fontsize=8, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

axes_flat[-1].axis('off')

plt.suptitle('Pixel-Level Metric Distributions by Ecoregion (Single Fire Events)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(eco_plots_dir, "statistical_tests_ecoregion_boxplots.png"), dpi=300, bbox_inches='tight')
plt.close()
print(f"  ✓ statistical_tests_ecoregion_boxplots.png")

# --- Dunn Heatmaps ---
if HAS_POSTHOC and dunn_results_eco:
    n_metrics_sig = len(dunn_results_eco)
    fig, axes_dunn = plt.subplots(1, n_metrics_sig, figsize=(7 * n_metrics_sig, 6))
    if n_metrics_sig == 1:
        axes_dunn = [axes_dunn]
    
    plot_i = 0
    for metric in METRICS_TO_TEST:
        if metric not in dunn_results_eco:
            continue
        ax = axes_dunn[plot_i]
        dunn_df = dunn_results_eco[metric]
        
        log_p = -np.log10(dunn_df.values.astype(float))
        log_p = np.clip(log_p, 0, 50)
        
        annot = dunn_df.copy()
        for r_idx in annot.index:
            for c_idx in annot.columns:
                p = dunn_df.loc[r_idx, c_idx]
                if r_idx == c_idx:
                    annot.loc[r_idx, c_idx] = ""
                elif p < 0.001:
                    annot.loc[r_idx, c_idx] = "***"
                elif p < 0.01:
                    annot.loc[r_idx, c_idx] = "**"
                elif p < 0.05:
                    annot.loc[r_idx, c_idx] = "*"
                else:
                    annot.loc[r_idx, c_idx] = "n.s."
        
        sns.heatmap(
            pd.DataFrame(log_p, index=dunn_df.index, columns=dunn_df.columns),
            cmap='RdYlGn_r', annot=annot.values, fmt='',
            ax=ax, square=True, linewidths=0.5,
            cbar_kws={'label': '-log₁₀(p)'}, vmin=0, vmax=10
        )
        ax.set_title(f'{METRIC_LABELS[metric]}\n(Dunn Post-hoc, Bonferroni)',
                     fontsize=10, fontweight='bold')
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
        plot_i += 1
    
    plt.suptitle("Pairwise Ecoregion Comparisons – Dunn's Post-hoc Test",
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(eco_plots_dir, "statistical_tests_ecoregion_dunn_heatmaps.png"),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ statistical_tests_ecoregion_dunn_heatmaps.png")

# === CSV EXPORTS ===
print("\nSpeichere Ecoregion-Statistik CSVs...")

# KW Zusammenfassung
kw_eco_rows = []
for metric in METRICS_TO_TEST:
    if metric in kw_results_eco:
        kw = kw_results_eco[metric]
        kw_eco_rows.append({
            'Metric': metric, 'Metric_Label': METRIC_LABELS[metric],
            'H_Statistic': kw['H_statistic'], 'p_value': kw['p_value'],
            'Eta_Squared': kw['eta_squared'], 'Effect_Size': kw['effect_size'],
            'k_Groups': kw['k_groups'], 'n_Total': kw['n_total'],
            'Significant_0.05': kw['significant']
        })

pd.DataFrame(kw_eco_rows).to_csv(
    os.path.join(eco_csv_dir, "statistical_tests_ecoregion_kruskal_wallis.csv"), index=False)

# Mediane pro Ecoregion
median_eco_rows = []
for eco_code in eco_names_ordered:
    row = {'Ecoregion': eco_code, 'N_Pixels_Sampled': eco_pixel_metrics[eco_code]['n_pixels_sampled']}
    for metric in METRICS_TO_TEST:
        vals = eco_pixel_metrics[eco_code][metric]
        row[f'{metric}_median'] = np.nanmedian(vals)
        row[f'{metric}_iqr'] = stats.iqr(vals[~np.isnan(vals)]) if np.sum(~np.isnan(vals)) > 0 else np.nan
    median_eco_rows.append(row)

pd.DataFrame(median_eco_rows).to_csv(
    os.path.join(eco_csv_dir, "statistical_tests_ecoregion_medians.csv"), index=False)

# Dunn Post-hoc p-Werte
if HAS_POSTHOC and dunn_results_eco:
    for metric, dunn_df in dunn_results_eco.items():
        dunn_df.to_csv(os.path.join(eco_csv_dir, f"statistical_tests_ecoregion_dunn_{metric}.csv"))

print(f"\n✓ Alle Ecoregion-Statistiken abgeschlossen!")
print(f"  Plots: {eco_plots_dir}")
print(f"  CSVs:  {eco_csv_dir}")



3d. STATISTISCHE TESTS – ECOREGION-VERGLEICHE
----------------------------------------------------------------------

Extrahiere Pixel-Level Metriken pro Ecoregion...


Ecoregions:   0%|          | 0/11 [00:00<?, ?it/s]

C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:61: RuntimeWarning: divide by zero encountered in divide
  fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:61: RuntimeWarning: invalid value encountered in divide
  fire_loss_pct = np.where(pre_fire_m1 > 0, fire_loss / pre_fire_m1 * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:63: RuntimeWarning: divide by zero encountered in divide
  (recovery_5yr_val - fire_year_val) / fire_loss * 100, np.nan)
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\1752821066.py:63: RuntimeWarning: invalid value encountered in divide
  (recovery_5yr_val - fire_year_val) / fire_loss * 100, np.nan)
Ecoregions:   9%|▉         | 1/11 [00:08<01:22,  8.24s/it]

  ✓ Anatolian: 74,752 Pixel (sampled: 50,000)


Ecoregions:  18%|█▊        | 2/11 [00:10<00:40,  4.55s/it]

...

Ecoregions: 100%|██████████| 11/11 [01:13<00:00,  6.68s/it]

  ✓ Arctic: 915 Pixel (sampled: 915)

  Pre-Fire Woody Cover (Mean, %):
    H = 130597.4, p = 0.00e+00 ***
    η² = 0.3036 (large)

  Pre-Fire Trend (Slope, %/yr):
    H = 7078.9, p = 0.00e+00 ***
    η² = 0.0164 (small)



  Absolute Fire Loss (%):
    H = 1573.9, p = 0.00e+00 ***
    η² = 0.0036 (negligible)

  Relative Fire Loss (%):
    H = 1293.8, p = 8.50e-272 ***
    η² = 0.0032 (negligible)

  5-Year Recovery (% of Loss):
    H = 3206.6, p = 0.00e+00 ***
    η² = 0.0301 (small)

  Dunn's Post-hoc Tests (Bonferroni) – Ecoregions:
    pre_fire_mean: 53/55 signifikante Paare
    pre_fire_trend: 47/55 signifikante Paare
    fire_loss: 43/55 signifikante Paare
    fire_loss_pct: 41/55 signifikante Paare
    recovery_5yr_pct: 46/55 signifikante Paare

Erstelle Ecoregion-Boxplots...
  ✓ statistical_tests_ecoregion_boxplots.png


C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\3534424608.py:142: RuntimeWarning: divide by zero encountered in log10
  log_p = -np.log10(dunn_df.values.astype(float))
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\3534424608.py:142: RuntimeWarning: divide by zero encountered in log10
  log_p = -np.log10(dunn_df.values.astype(float))
C:\Users\butzerfe\AppData\Local\Temp\20\ipykernel_123760\3534424608.py:142: RuntimeWarning: divide by zero encountered in log10
  log_p = -np.log10(dunn_df.values.astype(float))


  ✓ statistical_tests_ecoregion_dunn_heatmaps.png

Speichere Ecoregion-Statistik CSVs...

✓ Alle Ecoregion-Statistiken abgeschlossen!
  Plots: \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Ecoregions\plots
  CSVs:  \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Ecoregions\csv


Schritt 3d: Statistische Tests – Ecoregion
statistical_tests_ecoregion_boxplots.png

Figure 19. Box plot distributions of pixel-level fire impact metrics by ecoregion (single fire events). Metrics shown are: pre-fire mean woody cover, pre-fire trend, absolute and relative fire loss, and 5-year recovery. Boxes are coloured by ecoregion; outliers are suppressed. Kruskal–Wallis H-statistic, effect size (η²), and significance level are annotated per panel.

statistical_tests_ecoregion_dunn_heatmaps.png

Figure 20. Pairwise post-hoc comparisons (Dunn's test with Bonferroni correction) between ecoregions for each metric where the Kruskal–Wallis test was significant (p < 0.05). Cell colours represent −log₁₀(p); annotations indicate significance levels (*** p < 0.001, ** p < 0.01, * p < 0.05, n.s. = not significant).



In [33]:
# === SCHRITT 4: SPEICHERE ERGEBNISSE ===

print("\n4. SPEICHERE ERGEBNISSE")
print("-" * 70)

# === CSV 1: Trajektorien nach Major LC (1 Fire) → lc_csv_dir ===
rows = []
for lc_class, data in lc_major_results.items():
    for i, ry in enumerate(rel_years):
        rows.append({
            'LC_Major': lc_class,
            'N_Pixels': data['n_pixels'],
            'Rel_Year': ry,
            'Woody_Cover_Mean': data['trajectory'][i],
            'Woody_Cover_Std': data['std'][i]
        })
df_traj_major = pd.DataFrame(rows)
csv1 = os.path.join(lc_csv_dir, "trajectories_by_landcover_1fire.csv")
df_traj_major.to_csv(csv1, index=False)
print(f"  ✓ {csv1}")

# === CSV 2: Trajektorien nach LC × Fire Count → lc_csv_dir ===
rows = []
for lc_class, fc_data in lc_firecount_results.items():
    for fc, data in fc_data.items():
        for i, ry in enumerate(rel_years):
            rows.append({
                'LC_Major': lc_class,
                'Fire_Count': fc,
                'N_Pixels': data['n_pixels'],
                'Rel_Year': ry,
                'Woody_Cover_Mean': data['trajectory'][i],
                'Woody_Cover_Std': data['std'][i]
            })
df_traj_fc = pd.DataFrame(rows)
csv2 = os.path.join(lc_csv_dir, "trajectories_by_landcover_x_firecount.csv")
df_traj_fc.to_csv(csv2, index=False)
print(f"  ✓ {csv2}")

# === CSV 3: Trajektorien nach LC × Ecoregion → lcxeco_csv_dir ===
rows = []
for eco_code, lc_data in lc_eco_results.items():
    for lc_class, data in lc_data.items():
        for i, ry in enumerate(rel_years):
            rows.append({
                'Ecoregion': eco_code,
                'LC_Major': lc_class,
                'N_Pixels': data['n_pixels'],
                'Rel_Year': ry,
                'Woody_Cover_Mean': data['trajectory'][i],
                'Woody_Cover_Std': data['std'][i]
            })
df_traj_eco = pd.DataFrame(rows)
csv3 = os.path.join(lcxeco_csv_dir, "trajectories_by_landcover_x_ecoregion.csv")
df_traj_eco.to_csv(csv3, index=False)
print(f"  ✓ {csv3}")

# === CSV 4: Recovery Summary → lc_csv_dir ===
rows = []
for lc_class, data in lc_major_results.items():
    traj = np.array(data['trajectory'])
    pre_fire = np.nanmean(traj[:5])
    fire_year = traj[5]
    loss = pre_fire - fire_year
    
    for years_after in [1, 3, 5, 10]:
        idx = 5 + years_after
        if idx < len(traj):
            rec_val = traj[idx]
            rec_pct = ((rec_val - fire_year) / loss * 100) if loss > 0 else np.nan
        else:
            rec_val = np.nan
            rec_pct = np.nan
        
        rows.append({
            'LC_Major': lc_class,
            'N_Pixels': data['n_pixels'],
            'Pre_Fire_Cover': pre_fire,
            'Fire_Year_Cover': fire_year,
            'Woody_Loss': loss,
            'Years_After': years_after,
            'Recovery_Cover': rec_val,
            'Recovery_Percent': rec_pct
        })

df_recovery = pd.DataFrame(rows)
csv4 = os.path.join(lc_csv_dir, "recovery_summary_by_landcover.csv")
df_recovery.to_csv(csv4, index=False)
print(f"  ✓ {csv4}")


# === CSV 5: Trajektorien pro Ecoregion (1 Fire) → eco_csv_dir ===
rows = []
for eco_code, data in eco_results.items():
    for i, ry in enumerate(rel_years):
        rows.append({
            'Ecoregion': eco_code, 'Ecoregion_Name': data['name'],
            'N_Pixels': data['n_pixels'], 'Rel_Year': ry,
            'Woody_Cover_Mean': data['trajectory'][i],
            'Woody_Cover_Std': data['std'][i]
        })
csv5 = os.path.join(eco_csv_dir, "trajectories_by_ecoregion_1fire.csv")
pd.DataFrame(rows).to_csv(csv5, index=False)
print(f"  ✓ {csv5}")

# === CSV 6: Trajektorien Ecoregion × Fire Count → eco_csv_dir ===
rows = []
for eco_code, fc_data in eco_firecount_results.items():
    for fc, data in fc_data.items():
        for i, ry in enumerate(rel_years):
            rows.append({
                'Ecoregion': eco_code, 'Fire_Count': fc,
                'N_Pixels': data['n_pixels'], 'Rel_Year': ry,
                'Woody_Cover_Mean': data['trajectory'][i],
                'Woody_Cover_Std': data['std'][i]
            })
csv6 = os.path.join(eco_csv_dir, "trajectories_by_ecoregion_x_firecount.csv")
pd.DataFrame(rows).to_csv(csv6, index=False)
print(f"  ✓ {csv6}")

# === CSV 7: Burned Area Total → eco_csv_dir ===
csv7 = os.path.join(eco_csv_dir, "burned_area_annual_total.csv")
burned_area_total.to_csv(csv7, index=False)
print(f"  ✓ {csv7}")

# === CSV 8: Burned Area pro Ecoregion → eco_csv_dir ===
csv8 = os.path.join(eco_csv_dir, "burned_area_by_ecoregion.csv")
burned_area_eco.to_csv(csv8, index=False)
print(f"  ✓ {csv8}")

# === CSV 9: Fire Count Verteilung pro Ecoregion → eco_csv_dir ===
csv9 = os.path.join(eco_csv_dir, "fire_count_distribution_by_ecoregion.csv")
fire_count_eco.to_csv(csv9, index=False)
print(f"  ✓ {csv9}")

# === CSV 10: Season Length Stats → eco_csv_dir ===
csv10 = os.path.join(eco_csv_dir, "fire_season_length_by_ecoregion.csv")
season_stats.to_csv(csv10, index=False)
print(f"  ✓ {csv10}")

# === CSV 11: Fire Stats Summary → eco_csv_dir ===
csv11 = os.path.join(eco_csv_dir, "fire_statistics_summary_per_ecoregion.csv")
fire_stats_summary.to_csv(csv11, index=False)
print(f"  ✓ {csv11}")

# === CSV 12: Recovery pro Ecoregion → eco_csv_dir ===
csv12 = os.path.join(eco_csv_dir, "recovery_summary_by_ecoregion.csv")
df_eco_recovery.to_csv(csv12, index=False)
print(f"  ✓ {csv12}")

# === CSV 13: Summary Ecoregion × Fire Count → lcxeco_csv_dir ===
csv13 = os.path.join(lcxeco_csv_dir, "summary_ecoregion_x_firecount.csv")
df_summary.to_csv(csv13, index=False)
print(f"  ✓ {csv13}")

# === CSV 14: Season Length × Fire Count → eco_csv_dir === 
rows_sfc = []
for fc_val, fc_data in season_by_fc.items():
    rows_sfc.append({
        'Fire_Count': fc_val,
        'N_Pixels':   fc_data['n'],
        'Mean_Season_Length': fc_data['mean'],
    })
csv14 = os.path.join(eco_csv_dir, "season_length_by_fire_count.csv")
pd.DataFrame(rows_sfc).to_csv(csv14, index=False)
print(f"  ✓ {csv14}")


# === ABSCHLUSSMELDUNG ===
print("\n" + "=" * 80)
print("✓✓✓ LANDCOVER- & ECOREGION-STRATIFIZIERTE ANALYSE ABGESCHLOSSEN! ✓✓✓")
print("=" * 80)

print(f"\n📊 ANALYSIERTE DATEN:")
print(f"  Landcover-Typen: {len(lc_major_results)}")
print(f"  IGBP Einzelklassen: {len(lc_detail_results)}")
print(f"  LC × Fire Count Kombinationen: {sum(len(v) for v in lc_firecount_results.values())}")
print(f"  LC × Ecoregion Kombinationen: {sum(len(v) for v in lc_eco_results.values())}")
print(f"  Ecoregionen: {len(eco_results)}")
print(f"  Ecoregion × Fire Count Kombinationen: {sum(len(v) for v in eco_firecount_results.values())}")

print(f"\n📈 PLOTS (Landcover):")
print(f"  1. lc_trajectories_1fire_all.png        – Alle LC-Typen (1 Fire)")
print(f"  2. lc_trajectories_1fire_panel.png       – Subplots pro LC-Typ")
print(f"  3. lc_trajectories_by_fire_count.png     – LC × Fire Count")
print(f"  4. lc_ecoregion_heatmap.png              – Heatmap LC × Ecoregion")
print(f"  5. lc_trajectories_<ECO>.png             – Pro Ecoregion")
print(f"  6. statistical_tests_landcover_*.png     – Stat. Tests LC")

print(f"\n📈 PLOTS (Ecoregion):")
print(f"  E1. eco_trajectories_1fire_all.png       – Alle Ecoregionen (1 Fire)")
print(f"  E2. eco_trajectories_1fire_panel.png     – Subplots pro Ecoregion")
print(f"  E3. eco_trajectories_by_fire_count.png   – Eco × Fire Count")
print(f"  E4. burned_area_annual_total.png         – Jährl. verbrannte Fläche")
print(f"  E5. burned_area_by_ecoregion.png         – Burned Area pro Ecoregion")
print(f"  E6. fire_season_length_analysis.png      – Season Length (3 Panels)")
print(f"  E7. fire_count_analysis.png              – Fire Count (3 Panels)")
print(f"  E8. fire_season_length_by_count.png      – Season Length × Fire Count")
print(f"  E9. eco_recovery_heatmap.png             – Recovery Heatmap")
print(f"  E10. eco_summary_heatmap.png             – Summary Heatmap")
print(f"  E11. statistical_tests_ecoregion_*.png   – Stat. Tests Ecoregion")

print(f"\n📋 CSV-DATEIEN (Landcover) → {lc_csv_dir}")
print(f"   1. trajectories_by_landcover_1fire.csv")
print(f"   2. trajectories_by_landcover_x_firecount.csv")
print(f"   4. recovery_summary_by_landcover.csv")
print(f"   + statistical_tests_landcover_*.csv")

print(f"\n📋 CSV-DATEIEN (LC × Ecoregion) → {lcxeco_csv_dir}")
print(f"   3. trajectories_by_landcover_x_ecoregion.csv")
print(f"  13. summary_ecoregion_x_firecount.csv")
print(f"   + summary_statistics_by_fire_count.csv")

print(f"\n📋 CSV-DATEIEN (Ecoregion) → {eco_csv_dir}")
print(f"   5. trajectories_by_ecoregion_1fire.csv")
print(f"   6. trajectories_by_ecoregion_x_firecount.csv")
print(f"   7. burned_area_annual_total.csv")
print(f"   8. burned_area_by_ecoregion.csv")
print(f"   9. fire_count_distribution_by_ecoregion.csv")
print(f"  10. fire_season_length_by_ecoregion.csv")
print(f"  11. fire_statistics_summary_per_ecoregion.csv")
print(f"  12. recovery_summary_by_ecoregion.csv")
print(f"  14. season_length_by_fire_count.csv")
print(f"   + recovery_rates_by_fire_count_extended.csv")
print(f"   + statistical_tests_ecoregion_*.csv")
print("=" * 80)



4. SPEICHERE ERGEBNISSE
----------------------------------------------------------------------
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Landcover\csv\trajectories_by_landcover_1fire.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Landcover\csv\trajectories_by_landcover_x_firecount.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\LC_x_Ecoregion\csv\trajectories_by_landcover_x_ecoregion.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Landcover\csv\recovery_summary_by_landcover.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Ecoregions\csv\trajectories_by_ecoregion_1fire.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Ecoregions\csv\trajectories_by_ecoregion_x_firecount.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis\_Runs\09_Results\Ecoregions\csv\burned_area_annual_total.csv
  ✓ \\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\t

In [34]:
# === SCHNELLE VISUALISIERUNG AUS CSV (ohne Neuberechnung) ===
# Nutze diese Zelle, wenn du nur Plots anpassen willst

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"

# Output-Pfade (müssen mit Config-Zelle übereinstimmen)
_results_root   = os.path.join(workDir, "_Runs", "09_Results")
eco_plots_dir   = os.path.join(_results_root, "Ecoregions",     "plots")
eco_csv_dir     = os.path.join(_results_root, "Ecoregions",     "csv")
lc_plots_dir    = os.path.join(_results_root, "Landcover",      "plots")
lc_csv_dir      = os.path.join(_results_root, "Landcover",      "csv")
lcxeco_plots_dir = os.path.join(_results_root, "LC_x_Ecoregion", "plots")
lcxeco_csv_dir   = os.path.join(_results_root, "LC_x_Ecoregion", "csv")

# Konstanten
pixel_area_km2 = 0.25   # 500m × 500m
pixel_area_ha  = 25.0

# =====================================================
# A) LANDCOVER CSVs
# =====================================================
print("Lade Landcover CSVs...")

df_traj_major  = pd.read_csv(os.path.join(lc_csv_dir,    "trajectories_by_landcover_1fire.csv"))
df_traj_fc     = pd.read_csv(os.path.join(lc_csv_dir,    "trajectories_by_landcover_x_firecount.csv"))
df_traj_eco    = pd.read_csv(os.path.join(lcxeco_csv_dir, "trajectories_by_landcover_x_ecoregion.csv"))
df_recovery_lc = pd.read_csv(os.path.join(lc_csv_dir,    "recovery_summary_by_landcover.csv"))

print(f"  ✓ Major LC:     {len(df_traj_major):,} Zeilen")
print(f"  ✓ LC×FireCount: {len(df_traj_fc):,} Zeilen")
print(f"  ✓ LC×Ecoregion: {len(df_traj_eco):,} Zeilen")
print(f"  ✓ Recovery LC:  {len(df_recovery_lc):,} Zeilen")

# =====================================================
# B) ECOREGION CSVs
# =====================================================
print("\nLade Ecoregion CSVs...")

df_traj_eco_1fire = pd.read_csv(os.path.join(eco_csv_dir,   "trajectories_by_ecoregion_1fire.csv"))
df_traj_eco_fc    = pd.read_csv(os.path.join(eco_csv_dir,   "trajectories_by_ecoregion_x_firecount.csv"))
df_burned_total   = pd.read_csv(os.path.join(eco_csv_dir,   "burned_area_annual_total.csv"))
df_burned_eco     = pd.read_csv(os.path.join(eco_csv_dir,   "burned_area_by_ecoregion.csv"))
df_fire_count_eco = pd.read_csv(os.path.join(eco_csv_dir,   "fire_count_distribution_by_ecoregion.csv"))
df_season_stats   = pd.read_csv(os.path.join(eco_csv_dir,   "fire_season_length_by_ecoregion.csv"))
df_fire_summary   = pd.read_csv(os.path.join(eco_csv_dir,   "fire_statistics_summary_per_ecoregion.csv"))
df_recovery_eco   = pd.read_csv(os.path.join(eco_csv_dir,   "recovery_summary_by_ecoregion.csv"))
df_season_fc      = pd.read_csv(os.path.join(eco_csv_dir,   "season_length_by_fire_count.csv"))   # ← NEU (CSV 14)
df_summary_eco_fc = pd.read_csv(os.path.join(lcxeco_csv_dir, "summary_ecoregion_x_firecount.csv"))

print(f"  ✓ Eco 1Fire:       {len(df_traj_eco_1fire):,} Zeilen")
print(f"  ✓ Eco×FireCount:   {len(df_traj_eco_fc):,} Zeilen")
print(f"  ✓ Burned Total:    {len(df_burned_total):,} Zeilen")
print(f"  ✓ Burned Eco:      {len(df_burned_eco):,} Zeilen")
print(f"  ✓ Fire Count Eco:  {len(df_fire_count_eco):,} Zeilen")
print(f"  ✓ Season Stats:    {len(df_season_stats):,} Zeilen")
print(f"  ✓ Fire Summary:    {len(df_fire_summary):,} Zeilen")
print(f"  ✓ Recovery Eco:    {len(df_recovery_eco):,} Zeilen")
print(f"  ✓ Season×FC:       {len(df_season_fc):,} Zeilen")          # ← NEU
print(f"  ✓ Summary Eco×FC:  {len(df_summary_eco_fc):,} Zeilen")

# =====================================================
# C) ECOREGION-ATTRIBUTE LADEN (für Farben)
# =====================================================
print("\nLade Ecoregion-Attribute...")

ecoregion_attr_path = os.path.join(
    workDir,
    r"_Runs\03_Ecoregions_EEA\ecoregions_attributes_with_colors.csv"
)

ecoregion_df  = pd.read_csv(ecoregion_attr_path)
eco_color_map = dict(zip(ecoregion_df['code'], ecoregion_df['hex_color']))
eco_name_map  = dict(zip(ecoregion_df['code'], ecoregion_df['name']))

print(f"  ✓ {len(eco_color_map)} Ecoregion-Farben geladen")

# =====================================================
# D) REKONSTRUIERE DICTIONARIES
# =====================================================
print("\nRekonstruiere Dictionaries aus CSVs...")

rel_years = sorted(df_traj_major['Rel_Year'].unique())

# --- Landcover Dictionaries ---
lc_major_results = {}
for lc_class, grp in df_traj_major.groupby('LC_Major'):
    grp = grp.sort_values('Rel_Year')
    lc_major_results[lc_class] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std':        grp['Woody_Cover_Std'].tolist(),
        'n_pixels':   int(grp['N_Pixels'].iloc[0])
    }

lc_firecount_results = {}
for (lc_class, fc), grp in df_traj_fc.groupby(['LC_Major', 'Fire_Count']):
    grp = grp.sort_values('Rel_Year')
    if lc_class not in lc_firecount_results:
        lc_firecount_results[lc_class] = {}
    lc_firecount_results[lc_class][int(fc)] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std':        grp['Woody_Cover_Std'].tolist(),
        'n_pixels':   int(grp['N_Pixels'].iloc[0])
    }

lc_eco_results = {}
for (eco_code, lc_class), grp in df_traj_eco.groupby(['Ecoregion', 'LC_Major']):
    grp = grp.sort_values('Rel_Year')
    if eco_code not in lc_eco_results:
        lc_eco_results[eco_code] = {}
    lc_eco_results[eco_code][lc_class] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std':        grp['Woody_Cover_Std'].tolist(),
        'n_pixels':   int(grp['N_Pixels'].iloc[0])
    }

# --- Ecoregion Dictionaries ---
eco_results = {}
for eco_code, grp in df_traj_eco_1fire.groupby('Ecoregion'):
    grp = grp.sort_values('Rel_Year')
    eco_results[eco_code] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std':        grp['Woody_Cover_Std'].tolist(),
        'n_pixels':   int(grp['N_Pixels'].iloc[0]),
        'name':       grp['Ecoregion_Name'].iloc[0],
        'hex_color':  eco_color_map.get(eco_code, '#808080')
    }

eco_firecount_results = {}
for (eco_code, fc), grp in df_traj_eco_fc.groupby(['Ecoregion', 'Fire_Count']):
    grp = grp.sort_values('Rel_Year')
    if eco_code not in eco_firecount_results:
        eco_firecount_results[eco_code] = {}
    eco_firecount_results[eco_code][int(fc)] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std':        grp['Woody_Cover_Std'].tolist(),
        'n_pixels':   int(grp['N_Pixels'].iloc[0])
    }

# --- season_by_fc aus CSV 14 rekonstruieren ---
# ⚠️ 'values' (Rohdaten) nicht im CSV → Plot E8 nutzt nur mean/n (kein Boxplot möglich)
season_by_fc = {}
for _, row in df_season_fc.iterrows():
    fc_val = int(row['Fire_Count'])
    season_by_fc[fc_val] = {
        'n':      int(row['N_Pixels']),
        'mean':   float(row['Mean_Season_Length']),
        'values': np.array([])   # Rohdaten nicht im CSV → E8 Boxplot nicht verfügbar
    }

# --- Fire Stats Aliases ---
burned_area_total  = df_burned_total
burned_area_eco    = df_burned_eco
fire_count_eco     = df_fire_count_eco
season_stats       = df_season_stats
fire_stats_summary = df_fire_summary
df_eco_recovery    = df_recovery_eco
df_summary         = df_summary_eco_fc

# --- Farb-Konfigurationen ---
LC_COLORS = {
    'Forests':       '#228B22',
    'Shrublands':    '#8B4513',
    'Open Woodland': '#DAA520',
    'Grasslands':    '#90EE90',
    'Wetlands':      '#4682B4',
    'Croplands':     '#FFD700',
    'Barren/Ice':    '#D3D3D3'
}

FIRE_COUNT_COLORS = {
    1: '#1f77b4',
    2: '#ff7f0e',
    3: '#2ca02c',
    4: '#d62728'
}

# =====================================================
# ZUSAMMENFASSUNG
# =====================================================
print(f"\n✓ Alle Dictionaries rekonstruiert – Plots können direkt ausgeführt werden!")
print(f"\n  LANDCOVER:")
print(f"    {len(lc_major_results)} Major LC Klassen")
print(f"    {sum(len(v) for v in lc_firecount_results.values())} LC×FireCount Kombinationen")
print(f"    {sum(len(v) for v in lc_eco_results.values())} LC×Ecoregion Kombinationen")
print(f"\n  ECOREGION:")
print(f"    {len(eco_results)} Ecoregionen")
print(f"    {sum(len(v) for v in eco_firecount_results.values())} Eco×FireCount Kombinationen")
print(f"    {len(burned_area_total)} Jahre Burned Area (gesamt)")
print(f"    {len(burned_area_eco)} Eco×Jahr Burned Area Einträge")
print(f"    {len(fire_stats_summary)} Ecoregion Fire Statistics")
print(f"    {len(season_by_fc)} Fire-Count-Werte in season_by_fc")
print(f"\n  ⚠️  Plot E8 (Boxplot Season×FC) benötigt Rohdaten → nur bei vollem Run verfügbar!")
print(f"\n  → Führe Schritt 3 (LC-Plots) und 3c (Eco-Plots) direkt aus!")# === SCHNELLE VISUALISIERUNG AUS CSV (ohne Neuberechnung) ===
# Nutze diese Zelle, wenn du nur Plots anpassen willst

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"

# Output-Pfade (müssen mit Config-Zelle übereinstimmen)
_results_root   = os.path.join(workDir, "_Runs", "09_Results")
eco_plots_dir   = os.path.join(_results_root, "Ecoregions",     "plots")
eco_csv_dir     = os.path.join(_results_root, "Ecoregions",     "csv")
lc_plots_dir    = os.path.join(_results_root, "Landcover",      "plots")
lc_csv_dir      = os.path.join(_results_root, "Landcover",      "csv")
lcxeco_plots_dir = os.path.join(_results_root, "LC_x_Ecoregion", "plots")
lcxeco_csv_dir   = os.path.join(_results_root, "LC_x_Ecoregion", "csv")

# Konstanten
pixel_area_km2 = 0.25   # 500m × 500m
pixel_area_ha  = 25.0

# =====================================================
# A) LANDCOVER CSVs
# =====================================================
print("Lade Landcover CSVs...")

df_traj_major  = pd.read_csv(os.path.join(lc_csv_dir,    "trajectories_by_landcover_1fire.csv"))
df_traj_fc     = pd.read_csv(os.path.join(lc_csv_dir,    "trajectories_by_landcover_x_firecount.csv"))
df_traj_eco    = pd.read_csv(os.path.join(lcxeco_csv_dir, "trajectories_by_landcover_x_ecoregion.csv"))
df_recovery_lc = pd.read_csv(os.path.join(lc_csv_dir,    "recovery_summary_by_landcover.csv"))

print(f"  ✓ Major LC:     {len(df_traj_major):,} Zeilen")
print(f"  ✓ LC×FireCount: {len(df_traj_fc):,} Zeilen")
print(f"  ✓ LC×Ecoregion: {len(df_traj_eco):,} Zeilen")
print(f"  ✓ Recovery LC:  {len(df_recovery_lc):,} Zeilen")

# =====================================================
# B) ECOREGION CSVs
# =====================================================
print("\nLade Ecoregion CSVs...")

df_traj_eco_1fire = pd.read_csv(os.path.join(eco_csv_dir,   "trajectories_by_ecoregion_1fire.csv"))
df_traj_eco_fc    = pd.read_csv(os.path.join(eco_csv_dir,   "trajectories_by_ecoregion_x_firecount.csv"))
df_burned_total   = pd.read_csv(os.path.join(eco_csv_dir,   "burned_area_annual_total.csv"))
df_burned_eco     = pd.read_csv(os.path.join(eco_csv_dir,   "burned_area_by_ecoregion.csv"))
df_fire_count_eco = pd.read_csv(os.path.join(eco_csv_dir,   "fire_count_distribution_by_ecoregion.csv"))
df_season_stats   = pd.read_csv(os.path.join(eco_csv_dir,   "fire_season_length_by_ecoregion.csv"))
df_fire_summary   = pd.read_csv(os.path.join(eco_csv_dir,   "fire_statistics_summary_per_ecoregion.csv"))
df_recovery_eco   = pd.read_csv(os.path.join(eco_csv_dir,   "recovery_summary_by_ecoregion.csv"))
df_season_fc      = pd.read_csv(os.path.join(eco_csv_dir,   "season_length_by_fire_count.csv"))   # ← NEU (CSV 14)
df_summary_eco_fc = pd.read_csv(os.path.join(lcxeco_csv_dir, "summary_ecoregion_x_firecount.csv"))

print(f"  ✓ Eco 1Fire:       {len(df_traj_eco_1fire):,} Zeilen")
print(f"  ✓ Eco×FireCount:   {len(df_traj_eco_fc):,} Zeilen")
print(f"  ✓ Burned Total:    {len(df_burned_total):,} Zeilen")
print(f"  ✓ Burned Eco:      {len(df_burned_eco):,} Zeilen")
print(f"  ✓ Fire Count Eco:  {len(df_fire_count_eco):,} Zeilen")
print(f"  ✓ Season Stats:    {len(df_season_stats):,} Zeilen")
print(f"  ✓ Fire Summary:    {len(df_fire_summary):,} Zeilen")
print(f"  ✓ Recovery Eco:    {len(df_recovery_eco):,} Zeilen")
print(f"  ✓ Season×FC:       {len(df_season_fc):,} Zeilen")          # ← NEU
print(f"  ✓ Summary Eco×FC:  {len(df_summary_eco_fc):,} Zeilen")

# =====================================================
# C) ECOREGION-ATTRIBUTE LADEN (für Farben)
# =====================================================
print("\nLade Ecoregion-Attribute...")

ecoregion_attr_path = os.path.join(
    workDir,
    r"_Runs\03_Ecoregions_EEA\ecoregions_attributes_with_colors.csv"
)

ecoregion_df  = pd.read_csv(ecoregion_attr_path)
eco_color_map = dict(zip(ecoregion_df['code'], ecoregion_df['hex_color']))
eco_name_map  = dict(zip(ecoregion_df['code'], ecoregion_df['name']))

print(f"  ✓ {len(eco_color_map)} Ecoregion-Farben geladen")

# =====================================================
# D) REKONSTRUIERE DICTIONARIES
# =====================================================
print("\nRekonstruiere Dictionaries aus CSVs...")

rel_years = sorted(df_traj_major['Rel_Year'].unique())

# --- Landcover Dictionaries ---
lc_major_results = {}
for lc_class, grp in df_traj_major.groupby('LC_Major'):
    grp = grp.sort_values('Rel_Year')
    lc_major_results[lc_class] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std':        grp['Woody_Cover_Std'].tolist(),
        'n_pixels':   int(grp['N_Pixels'].iloc[0])
    }

lc_firecount_results = {}
for (lc_class, fc), grp in df_traj_fc.groupby(['LC_Major', 'Fire_Count']):
    grp = grp.sort_values('Rel_Year')
    if lc_class not in lc_firecount_results:
        lc_firecount_results[lc_class] = {}
    lc_firecount_results[lc_class][int(fc)] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std':        grp['Woody_Cover_Std'].tolist(),
        'n_pixels':   int(grp['N_Pixels'].iloc[0])
    }

lc_eco_results = {}
for (eco_code, lc_class), grp in df_traj_eco.groupby(['Ecoregion', 'LC_Major']):
    grp = grp.sort_values('Rel_Year')
    if eco_code not in lc_eco_results:
        lc_eco_results[eco_code] = {}
    lc_eco_results[eco_code][lc_class] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std':        grp['Woody_Cover_Std'].tolist(),
        'n_pixels':   int(grp['N_Pixels'].iloc[0])
    }

# --- Ecoregion Dictionaries ---
eco_results = {}
for eco_code, grp in df_traj_eco_1fire.groupby('Ecoregion'):
    grp = grp.sort_values('Rel_Year')
    eco_results[eco_code] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std':        grp['Woody_Cover_Std'].tolist(),
        'n_pixels':   int(grp['N_Pixels'].iloc[0]),
        'name':       grp['Ecoregion_Name'].iloc[0],
        'hex_color':  eco_color_map.get(eco_code, '#808080')
    }

eco_firecount_results = {}
for (eco_code, fc), grp in df_traj_eco_fc.groupby(['Ecoregion', 'Fire_Count']):
    grp = grp.sort_values('Rel_Year')
    if eco_code not in eco_firecount_results:
        eco_firecount_results[eco_code] = {}
    eco_firecount_results[eco_code][int(fc)] = {
        'trajectory': grp['Woody_Cover_Mean'].tolist(),
        'std':        grp['Woody_Cover_Std'].tolist(),
        'n_pixels':   int(grp['N_Pixels'].iloc[0])
    }

# --- season_by_fc aus CSV 14 rekonstruieren ---
# ⚠️ 'values' (Rohdaten) nicht im CSV → Plot E8 nutzt nur mean/n (kein Boxplot möglich)
season_by_fc = {}
for _, row in df_season_fc.iterrows():
    fc_val = int(row['Fire_Count'])
    season_by_fc[fc_val] = {
        'n':      int(row['N_Pixels']),
        'mean':   float(row['Mean_Season_Length']),
        'values': np.array([])   # Rohdaten nicht im CSV → E8 Boxplot nicht verfügbar
    }

# --- Fire Stats Aliases ---
burned_area_total  = df_burned_total
burned_area_eco    = df_burned_eco
fire_count_eco     = df_fire_count_eco
season_stats       = df_season_stats
fire_stats_summary = df_fire_summary
df_eco_recovery    = df_recovery_eco
df_summary         = df_summary_eco_fc

# --- Farb-Konfigurationen ---
LC_COLORS = {
    'Forests':       '#228B22',
    'Shrublands':    '#8B4513',
    'Open Woodland': '#DAA520',
    'Grasslands':    '#90EE90',
    'Wetlands':      '#4682B4',
    'Croplands':     '#FFD700',
    'Barren/Ice':    '#D3D3D3'
}

FIRE_COUNT_COLORS = {
    1: '#1f77b4',
    2: '#ff7f0e',
    3: '#2ca02c',
    4: '#d62728'
}

# =====================================================
# ZUSAMMENFASSUNG
# =====================================================
print(f"\n✓ Alle Dictionaries rekonstruiert – Plots können direkt ausgeführt werden!")
print(f"\n  LANDCOVER:")
print(f"    {len(lc_major_results)} Major LC Klassen")
print(f"    {sum(len(v) for v in lc_firecount_results.values())} LC×FireCount Kombinationen")
print(f"    {sum(len(v) for v in lc_eco_results.values())} LC×Ecoregion Kombinationen")
print(f"\n  ECOREGION:")
print(f"    {len(eco_results)} Ecoregionen")
print(f"    {sum(len(v) for v in eco_firecount_results.values())} Eco×FireCount Kombinationen")
print(f"    {len(burned_area_total)} Jahre Burned Area (gesamt)")
print(f"    {len(burned_area_eco)} Eco×Jahr Burned Area Einträge")
print(f"    {len(fire_stats_summary)} Ecoregion Fire Statistics")
print(f"    {len(season_by_fc)} Fire-Count-Werte in season_by_fc")
print(f"\n  ⚠️  Plot E8 (Boxplot Season×FC) benötigt Rohdaten → nur bei vollem Run verfügbar!")
print(f"\n  → Führe Schritt 3 (LC-Plots) und 3c (Eco-Plots) direkt aus!")

Lade Landcover CSVs...
  ✓ Major LC:     96 Zeilen
  ✓ LC×FireCount: 384 Zeilen
  ✓ LC×Ecoregion: 880 Zeilen
  ✓ Recovery LC:  24 Zeilen

Lade Ecoregion CSVs...
  ✓ Eco 1Fire:       176 Zeilen
  ✓ Eco×FireCount:   656 Zeilen
  ✓ Burned Total:    25 Zeilen
  ✓ Burned Eco:      275 Zeilen
  ✓ Fire Count Eco:  164 Zeilen
  ✓ Season Stats:    11 Zeilen
  ✓ Fire Summary:    11 Zeilen
  ✓ Recovery Eco:    44 Zeilen
  ✓ Season×FC:       4 Zeilen
  ✓ Summary Eco×FC:  41 Zeilen

Lade Ecoregion-Attribute...
  ✓ 11 Ecoregion-Farben geladen

Rekonstruiere Dictionaries aus CSVs...

✓ Alle Dictionaries rekonstruiert – Plots können direkt ausgeführt werden!

  LANDCOVER:
    6 Major LC Klassen
    24 LC×FireCount Kombinationen
    55 LC×Ecoregion Kombinationen

  ECOREGION:
    11 Ecoregionen
    41 Eco×FireCount Kombinationen
    25 Jahre Burned Area (gesamt)
    275 Eco×Jahr Burned Area Einträge
    11 Ecoregion Fire Statistics
    4 Fire-Count-Werte in season_by_fc

  ⚠️  Plot E8 (Boxplot Season×F